In [1]:
import pandas as pd

In [2]:
production_sample = pd.read_csv("../data/raw/produccion-resume.csv")

In [3]:
fracture_path = "../data/raw/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv"
fracture_df = pd.read_csv(fracture_path)

# 1. Notebook Overview

## Objective

Prepare and integrate the production and hydraulic fracturing datasets in MySQL to create a clean and consistent analytical database for subsequent SQL, Python, and Power BI analyses.

## Overview

Following the data audit and business understanding stages, this notebook focuses on preparing the datasets for analysis.

The workflow includes importing the official datasets into MySQL, cleaning and validating the data, integrating both datasets through the well identifier (`idpozo`), and creating an analytical dataset optimized for business analysis.

# 2. Database Design

In [4]:
comparison = (
    production_sample[
        ["idpozo", "formprod", "formacion"]
    ]
    .drop_duplicates()
    .merge(
        fracture_df[
            ["idpozo", "formacion_productiva"]
        ].drop_duplicates(),
        on="idpozo",
        how="inner"
    )
)

comparison.head(20)

,idpozo,formprod,formacion,formacion_productiva
0,130518,PREC,precuyo,precuyo
1,146139,LAJA,lajas,lajas
2,157213,LAJA,lajas,lajas
3,130158,PREC,precuyo,precuyo
4,131344,PREC,precuyo,precuyo
5,156620,LAJA,lajas,lajas
6,153801,LAJA,lajas,lajas
7,155584,LAJA,lajas,lajas
8,153363,LAJA,lajas,lajas
9,145327,LAJA,lajas,lajas


In [5]:
comparison[
    comparison["formacion"] != comparison["formacion_productiva"]
]

,idpozo,formprod,formacion,formacion_productiva


#### Data Modeling Decision:

The production dataset contains both a geological formation code (`formprod`) and its descriptive name (`formacion`), while the hydraulic fracturing dataset contains the descriptive field `formacion_productiva`. A comparison of shared wells showed that formacion and formacion_productiva are consistent, supporting the creation of a single shared dim_geology dimension. The production code (formprod) will be retained as an additional attribute of this dimension.

# 3. Geographical Hierarchy Validation

#### Objective

Validate the relationships between the geographical variables to support the dimensional model design.

#### 1. Basin → Province

In [6]:
production_sample.groupby("cuenca")["provincia"].nunique()

cuenca
GOLFO SAN JORGE    2
NEUQUINA           3
Name: provincia, dtype: int64

#### Finding

The results show that a basin may be associated with more than one province.

In [7]:
production_sample.groupby("cuenca")["provincia"].unique()

cuenca
GOLFO SAN JORGE             [Chubut, Santa Cruz]
NEUQUINA           [Neuquén, Rio Negro, Mendoza]
Name: provincia, dtype: object

#### Finding

Examples such as:

- Neuquina Basin → Neuquén, Río Negro, Mendoza
- Golfo San Jorge Basin → Chubut, Santa Cruz

confirm that a basin can extend across multiple provinces.

#### Conclusion

Province is not hierarchically determined by Basin. Therefore, these variables should not be modeled as a strict geographical hierarchy.

#### 2. License Area → Basin

In [8]:
production_sample.groupby("areapermisoconcesion")["cuenca"].nunique()

areapermisoconcesion
AGUADA DE LA ARENA               1
AGUADA DEL CHAÑAR                1
AL NORTE DE LA DORSAL            1
AL SUR DE LA DORSAL              1
ANTICLINAL CAMPAMENTO            1
BAJADA DE AÑELO                  1
BAJO DEL TORO NORTE              1
BANDURRIA SUR                    1
CAÑADON YATEL                    1
CERRO ARENA                      1
CERRO LAS MINAS                  1
CERRO PARTIDO                    1
CHIHUIDO DE LA SIERRA NEGRA      1
CORTADERA                        1
DADIN                            1
EL OREJANO                       1
ESCALANTE - EL TREBOL            1
ESTACION FERNANDEZ ORO           1
FILO MORADO                      1
HUACALERA                        1
LA AMARGA CHICA                  1
LA ANGOSTURA SUR I               1
LA ANGOSTURA SUR II              1
LA RIBERA BLOQUE I               1
LA RIBERA BLOQUE II              1
LAS TACANAS I                    1
LAS TACANAS II                   1
LOMA AMARILLA NORTE              1

#### Finding

Each License Area is associated with only one Basin.

#### Conclusion

License Area has a consistent one-to-many relationship with Basin and can be considered a valid hierarchical level.

#### 3. Oil Field → License Area

In [9]:
production_sample.groupby("areayacimiento")["areapermisoconcesion"].nunique()

areayacimiento
AGUADA DE LA ARENA             1
ANTICLINAL CAMPAMENTO          1
ANTICLINAL CAMPAMENTO OESTE    1
ANTICLINAL CAMPAMENTO SUR      1
BAJADA DE AÑELO                1
BAJO DEL TORO NORTE            1
BANDURRIA SUR                  1
BARDA GONZALEZ                 1
BARREALES COLORADOS            1
BARROSA NORTE                  1
CERRO ARENA                    1
CERRO LAS MINAS                1
CERRO PARTIDO                  1
CHIHUIDO DE LA SIERRA NEGRA    1
CORTADERA                      1
CUPEN MAHUIDA                  1
EL CORDON OESTE                1
EL OREJANO                     1
EL TREBOL                      1
EL TRIANGULO                   1
ESTACION FERNANDEZ ORO         1
ESTANCIA CHOLITA               1
FILO MORADO                    1
GUANACO                        1
HUACALERA                      1
HUINCUL                        1
LA AMARGA CHICA                1
LA ANGOSTURA SUR I             1
LA ANGOSTURA SUR II            1
LA RIBERA BLOQUE I          

#### Finding

Each Oil Field belongs to a single License Area.

#### Conclusion

The relationship between Oil Field and License Area is consistent and supports the geographical hierarchy.

#### 4. Well → Province

In [10]:
production_sample.groupby("idpozo")["provincia"].nunique()

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
         ..
166202    1
166203    1
166204    1
166205    1
166206    1
Name: provincia, Length: 1534, dtype: int64

#### Finding

Each well is associated with only one Province.

#### 5. Well → Basin

In [11]:
production_sample.groupby("idpozo")["cuenca"].nunique()

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
         ..
166202    1
166203    1
166204    1
166205    1
166206    1
Name: cuenca, Length: 1534, dtype: int64

#### Finding

Each well belongs to only one Basin.

#### 6. Well → License Area

In [12]:
production_sample.groupby("idpozo")["areapermisoconcesion"].nunique()

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
         ..
166202    1
166203    1
166204    1
166205    1
166206    1
Name: areapermisoconcesion, Length: 1534, dtype: int64

#### Finding

Each well belongs to only one License Area.

#### 7. Well → Oil Field

In [13]:
production_sample.groupby("idpozo")["areayacimiento"].nunique()

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
         ..
166202    1
166203    1
166204    1
166205    1
166206    1
Name: areayacimiento, Length: 1534, dtype: int64

#### Finding

Each well belongs to only one Oil Field.

### Final Conclusion

The validation confirms that geographical information represents static attributes of each well.

Although a Basin may span multiple Provinces, every well is consistently associated with a unique geographical location (Province, Basin, License Area, and Oil Field).

These findings support modeling the geographical information as a dimension linked to dim_well rather than storing location attributes directly in the fact tables.

# 4. Event Granularity Validation

#### Objective

Determine whether the hydraulic fracturing dataset is recorded at the well level or at the fracturing event level. This validation is required to correctly identify the grain of the fact table and decide which attributes belong to the well dimension and which should remain in the fracturing fact table.

In [14]:
fracture_df["idpozo"].value_counts().head(20)

idpozo
159341    6
136856    5
128039    4
133431    4
145328    4
155714    4
162702    4
135444    4
135463    4
161465    4
152779    4
147505    4
155092    3
136581    3
135445    3
144590    3
159910    3
145970    3
137884    3
153207    3
Name: count, dtype: int64

#### Finding

Several wells appear multiple times in the hydraulic fracturing dataset. For example, some wells have between three and six associated records.

#### Conclusion

The dataset is not unique at the well level, indicating that a single well may have multiple hydraulic fracturing records. However, additional validation is required to determine whether these records represent independent fracturing events or another level of operational detail.

## Well-Level Fracturing Record Inspection

#### Objective

Inspect wells with multiple hydraulic fracturing records to determine the real grain of the fracturing dataset and identify whether repeated records represent different operations or multiple stages within the same operation

In [15]:
fracture_df.loc[
    fracture_df["idpozo"] == 159341,
    [
        "id_base_fractura_adjiv",
        "fecha_inicio_fractura",
        "fecha_fin_fractura",
        "cantidad_fracturas",
        "longitud_rama_horizontal_m",
        "presion_maxima_psi",
        "arena_bombeada_nacional_tn",
        "agua_inyectada_m3"
    ]
]

,id_base_fractura_adjiv,fecha_inicio_fractura,fecha_fin_fractura,cantidad_fracturas,longitud_rama_horizontal_m,presion_maxima_psi,arena_bombeada_nacional_tn,agua_inyectada_m3
825,1151,2018-05-28,2018-06-02,5,0.0,4500.0,0.0,396.4
2346,2926,2018-05-28,2018-06-17,1,0.0,4085.0,0.0,76.0
2347,2927,2018-05-28,2018-06-17,1,0.0,4125.0,0.0,50.0
2348,2928,2018-05-28,2018-06-17,1,0.0,3800.0,0.0,90.0
2349,2929,2018-05-28,2018-06-17,1,0.0,3200.0,0.0,130.0
2350,2930,2018-05-28,2018-06-17,1,0.0,2820.0,0.0,125.0


The analysis confirms that hydraulic fracturing records are not attributes of the well itself. Instead, they represent individual fracturing operations or stages associated with a well.

Therefore, operational variables such as fracture count, pressure, injected fluids, and proppant volumes should remain within the fracturing fact table (`fact_fracturing`) rather than being stored in the well dimension (`dim_well`).

In [16]:
fracture_df[
    fracture_df["idpozo"] == 159341
][
    [
        "id_base_fractura_adjiv",
        "fecha_inicio_fractura",
        "fecha_fin_fractura"
    ]
]

,id_base_fractura_adjiv,fecha_inicio_fractura,fecha_fin_fractura
825,1151,2018-05-28,2018-06-02
2346,2926,2018-05-28,2018-06-17
2347,2927,2018-05-28,2018-06-17
2348,2928,2018-05-28,2018-06-17
2349,2929,2018-05-28,2018-06-17
2350,2930,2018-05-28,2018-06-17


#### Finding

The well idpozo = 159341 contains six hydraulic fracturing records associated with different id_base_fractura_adjiv values.

However, all records share the same fracturing start date:

fecha_inicio_fractura = 2018-05-28

Several records also share the same end date:

fecha_fin_fractura = 2018-06-17

In [17]:
fracture_df[
    fracture_df["idpozo"] == 159341
]["fecha_inicio_fractura"].nunique()

1

#### Finding

The well has only one unique fracturing start date, suggesting that the multiple records may correspond to different fracturing stages or operational records within the same fracturing campaign rather than independent fracturing events.

#### Conclusion

The hydraulic fracturing dataset does not have a well-level grain.

Although multiple records can belong to the same well, the records contain operational attributes that vary between entries, such as:

- fracture identifiers,
- fracture stages,
- injected volumes,
- pressure values,
- proppant quantities.

Therefore, these variables should remain in a dedicated fracturing fact table and should not be incorporated into dim_well.

In [18]:
fracture_df[
    fracture_df["idpozo"] == 159341
][
    [
        "id_base_fractura_adjiv",
        "cantidad_fracturas",
        "longitud_rama_horizontal_m",
        "presion_maxima_psi",
        "arena_bombeada_nacional_tn",
        "agua_inyectada_m3"
    ]
]

,id_base_fractura_adjiv,cantidad_fracturas,longitud_rama_horizontal_m,presion_maxima_psi,arena_bombeada_nacional_tn,agua_inyectada_m3
825,1151,5,0.0,4500.0,0.0,396.4
2346,2926,1,0.0,4085.0,0.0,76.0
2347,2927,1,0.0,4125.0,0.0,50.0
2348,2928,1,0.0,3800.0,0.0,90.0
2349,2929,1,0.0,3200.0,0.0,130.0
2350,2930,1,0.0,2820.0,0.0,125.0


#### Conclusion

The analysis confirms that hydraulic fracturing records represent a lower level of detail than the well entity.

The variables related to fracturing operations, including:

number of fracture stages,
maximum pressure,
injected water volume,
proppant volumes,
operational parameters,

should remain in a dedicated fact_fracturing table.

They should not be stored in dim_well, because a single well can have multiple associated fracturing records with different operational characteristics.

# 5. Fracturing Record Identifier Validation

#### Objective

Validate whether `id_base_fractura_adjiv` represents a unique identifier for hydraulic fracturing records.

This validation is required to determine whether this field can be used as the primary key of the future fact_fracturing table.

In [19]:
fracture_df["id_base_fractura_adjiv"].nunique()

4806

In [20]:
len(fracture_df)

4806

In [21]:
fracture_df["id_base_fractura_adjiv"].duplicated().sum()

np.int64(0)

#### Finding

The hydraulic fracturing dataset contains 4,806 records and 4,806 unique values of `id_base_fractura_adjiv`.

No duplicated identifiers were found, confirming that each hydraulic fracturing record has a unique identifier.

The validated grain of the table is:

One row represents one hydraulic fracturing record identified by id_base_fractura_adjiv associated with a specific well (idpozo).

# 6. Production Record Granularity Validation

#### Objective

Validate the grain of the production dataset and identify the natural key that uniquely defines each production record.

This validation will determine whether each row represents a unique well production measurement over a specific time period.

In [22]:
production_sample.duplicated().sum()

np.int64(0)

#### Finding

No fully duplicated records were found in the production dataset.

In [23]:
production_sample[
    ["idpozo", "anio", "mes"]
].duplicated().sum()

np.int64(0)

#### Finding

No duplicated combinations of Well–Year–Month were found.

This suggests that each production record corresponds to a unique well during a specific reporting period.

In [24]:
production_sample["idpozo"].nunique()

1534

In [25]:
len(production_sample)

10000

#### Finding

The sample contains 10,000 production records corresponding to 1,534 unique wells.

This indicates that each well is associated with multiple production records over time.

In [26]:
production_sample.groupby("idpozo")[["anio","mes"]].count().head()

,anio,mes
idpozo,,
72206,1,1
72228,1,1
72232,1,1
75604,1,1
75605,1,1


In [27]:
production_sample["idpozo"].value_counts().head(20)

idpozo
137212    45
137716    45
137602    45
137718    45
137600    45
135586    45
136721    45
136722    45
135583    45
137599    45
136580    45
137601    45
134010    45
137074    45
153073    45
136579    45
137096    45
137073    45
152923    45
137213    45
Name: count, dtype: int64

#### Finding

Several wells appear repeatedly in the production dataset. The highest frequency observed in the sample is 45 production records for a single well.

This behavior is expected because production is recorded periodically over time rather than as a single measurement per well.

In [28]:
production_sample["idpozo"].value_counts().describe()

count    1534.000000
mean        6.518905
std        12.742749
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max        45.000000
Name: count, dtype: float64

#### Finding

The distribution of production records per well shows considerable variation.

Unique wells: 1,534
Average records per well: 6.5
Median: 2
Maximum: 45

This indicates that some wells have long production histories, while others contain only a few recorded periods within the analyzed sample.

#### Conclusion

The production dataset is recorded at the well-period level rather than at the well level.

Each record represents the production measurements of a specific well during a unique reporting period defined by year and month.

The validation confirmed that the combination (idpozo, anio, mes) uniquely identifies every production record in the analyzed sample, making it the natural grain of the future fact_production table.

# 7. Dimensional Model Design

#### Objective

Design a dimensional data model based on the business requirements and the data validation performed in the previous sections.

The objective is to organize the datasets into fact and dimension tables that support efficient SQL queries, business analysis, and dashboard development while preserving the natural granularity of the available data.

#### Overview

The validation process revealed that the project involves two distinct business processes, each operating at a different level of granularity:

- Monthly well production.
- Hydraulic fracturing operations.

Therefore, the proposed dimensional model consists of two fact tables supported by a shared set of dimension tables.

The proposed star schema includes:

- `fact_production`
- `fact_fracturing`
- `dim_well`
- `dim_time`
- `dim_geology`
- `dim_location`
- `dim_company`

# 8. Star Schema Overview

| Table | Type | Grain | Primary Key |
|-------|------|-------|-------------|
| `fact_production` | Fact | One row per well per month | (`idpozo`, `anio`, `mes`) |
| `fact_fracturing` | Fact | One row per hydraulic fracturing record | `id_base_fractura_adjiv` |
| `dim_well` | Dimension | One row per well | `idpozo` |
| `dim_time` | Dimension | One row per reporting period | `time_id` |
| `dim_geology` | Dimension | One row per geological entity | `geology_id` |
| `dim_location` | Dimension | One row per location | `location_id` |
| `dim_company` | Dimension | One row per company | `company_id` |

# 9. Star Schema Design

#### Objective

Define the relationships between the fact and dimension tables before designing the physical database schema.

```mermaid
erDiagram

    DIM_COMPANY ||--o{ DIM_WELL : owns
    DIM_LOCATION ||--o{ DIM_WELL : located_in
    DIM_GEOLOGY ||--o{ DIM_WELL : belongs_to

    DIM_WELL ||--o{ FACT_PRODUCTION : produces
    DIM_TIME ||--o{ FACT_PRODUCTION : recorded_at

    DIM_WELL ||--o{ FACT_FRACTURING : fractured
    DIM_TIME ||--o{ FACT_FRACTURING : occurred_at
```

| Relationship | Cardinality | Validated |
|--------------|-------------|-----------|
| Company → Well | 1:N | ✅ |
| Location → Well |	1:N | ✅ |
| Geology → Well | 1:N | ✅ |
| Well → Production | 1:N |	✅ |
| Well → Fracturing	| 1:N |	✅ |
| Time → Production	| 1:N |	✅ |
| Time → Fracturing	| 1:N |	✅ |

## 9.1 Temporal Columns Audit

#### Notebook Overview

This section evaluates every temporal attribute available in the production and fracturing datasets. The objective is to identify business dates, administrative dates, redundant temporal fields, and determine which attributes should participate in the Time Dimension (dim_time) and the fact tables.

#### Objective

Evaluate all temporal columns present in both source datasets to:

- identify their business meaning,
- detect redundant attributes,
- distinguish operational dates from administrative dates,
- define the temporal keys required for the dimensional model.

In [29]:
production_sample[["anio", "mes", "fecha_data"]].head()

,anio,mes,fecha_data
0,2015,1,2015-01-31
1,2017,1,2017-01-31
2,2015,1,2015-01-31
3,2018,1,2018-01-31
4,2015,1,2015-01-31


In [30]:
production_sample[["anio", "mes", "fecha_data"]].dtypes

anio           int64
mes            int64
fecha_data    object
dtype: object

In [31]:
fracture_df[["fecha_inicio_fractura", "fecha_fin_fractura", "fecha_data", "anio_if", "mes_if", "anio_ff", "anio_carga", "mes_carga", "mes", "anio"]].head()

,fecha_inicio_fractura,fecha_fin_fractura,fecha_data,anio_if,mes_if,anio_ff,anio_carga,mes_carga,mes,anio
0,2019-04-20,2019-04-30,2019-06-14 17:13:03.68279,2019,4,2019,2019,6,4,2019
1,2018-11-02,2018-11-03,2019-06-14 17:14:19.179874,2018,11,2018,2019,6,11,2018
2,2017-11-19,2017-12-14,2019-06-27 13:46:21.14935,2017,11,2017,2019,6,11,2017
3,2017-11-21,2017-12-15,2019-06-27 13:46:21.14935,2017,11,2017,2019,6,11,2017
4,2017-11-18,2017-12-15,2019-06-27 13:46:21.14935,2017,11,2017,2019,6,11,2017


In [32]:
fracture_df[["fecha_inicio_fractura", "fecha_fin_fractura", "fecha_data", "anio_if", "mes_if", "anio_ff", "anio_carga", "mes_carga", "mes", "anio"]].dtypes

fecha_inicio_fractura    object
fecha_fin_fractura       object
fecha_data               object
anio_if                   int64
mes_if                    int64
anio_ff                   int64
anio_carga                int64
mes_carga                 int64
mes                       int64
anio                      int64
dtype: object

#### Findings

- production_sample contains a single business time reference represented by the combination of anio and mes. Additionally, fecha_data corresponds to the last day of each production month, suggesting it represents the consolidated reporting date for the monthly production record.

- fracture_df contains three different categories of temporal information:

    - Business dates: fecha_inicio_fractura and fecha_fin_fractura, representing the actual start and end dates of the hydraulic fracturing operation.

    - Administrative date: fecha_data, representing the date when the record was loaded or registered in the source system.
    
    - Derived temporal attributes: anio_if, mes_if, anio_ff, anio, mes, anio_carga, and mes_carga, which store individual components extracted from the complete dates.

- All temporal attributes present consistent data types (int64 for date components and object for complete dates). The complete date fields should later be converted to datetime during the ETL process before loading the dimensional model.

- A clear redundancy exists between complete dates (fecha_inicio_fractura, fecha_fin_fractura) and their derived year/month attributes, since these components can be generated directly from the original dates.

- The coexistence of business dates and administrative dates indicates that not every temporal attribute should participate in the Time Dimension. Business dates should drive analytical relationships, whereas administrative dates are primarily intended for data lineage and audit purposes.

### Conclusion

The temporal audit confirms that both datasets contain sufficient information to support the construction of a shared Time Dimension (dim_time).

However, before defining the final dimensional model, each temporal attribute should be evaluated according to its business meaning. Business event dates will be used as the primary relationships with the fact tables, while administrative and derived temporal attributes should be excluded or generated dynamically to avoid redundancy.

### 9.1.1 Business Interpretation — `fecha_data` (production)

#### Objective

Determine whether `fecha_data` represents the appropriate business date for the Production Fact Table and evaluate whether it should be used as the relationship key with the Time Dimension (dim_time).

In [33]:
production_sample["fecha_data"].dtype

dtype('O')

In [34]:
pd.to_datetime(production_sample["fecha_data"]).head()

0   2015-01-31
1   2017-01-31
2   2015-01-31
3   2018-01-31
4   2015-01-31
Name: fecha_data, dtype: datetime64[ns]

In [35]:
fecha = pd.to_datetime(production_sample["fecha_data"])
(
    fecha.dt.day
    ==
    fecha.dt.days_in_month
).value_counts()

fecha_data
True    10000
Name: count, dtype: int64

In [36]:
(
    (fecha.dt.year == production_sample["anio"])
    &
    (fecha.dt.month == production_sample["mes"])
).value_counts()

True    10000
Name: count, dtype: int64

In [37]:
fecha.min(), fecha.max()

(Timestamp('2015-01-31 00:00:00'), Timestamp('2026-01-31 00:00:00'))

#### Findings

- The `fecha_data` column is currently stored as a string (object) but can be converted to a valid datetime type without inconsistencies.

- Every record corresponds exactly to the last calendar day of its respective month, confirming that the field represents the monthly reporting period rather than an arbitrary date.

- The year and month extracted from fecha_data match the values stored in anio and mes for 100% of the records, demonstrating that these attributes are fully consistent and represent the same business period.

- The temporal coverage ranges from January 2015 to January 2026, providing a continuous business timeline suitable for a shared Time Dimension.

- Since fecha_data fully encapsulates the information contained in anio and mes, those two attributes become functionally redundant once a proper date dimension is implemented.

### Conclusion

The audit confirms that `fecha_data` represents the business reporting date of each monthly production record.

Although production is reported at a monthly granularity, using a complete date (the last day of each reporting month) provides a standardized temporal reference that can be directly linked to a shared dim_time.

Therefore, fecha_data should be selected as the primary temporal key for fact_production, while anio and mes should be considered derived attributes obtained from the Time Dimension rather than independent keys.

### 9.1.2 Business Interpretation — `fecha_inicio_fractura`

#### Objective

Evaluate whether `fecha_inicio_fractura` represents the primary business date of each hydraulic fracturing operation and determine whether it should be used as the temporal key linking fact_fracturing to the shared Time Dimension (dim_time).

In [38]:
fracture_df["fecha_inicio_fractura"].dtype

dtype('O')

In [39]:
pd.to_datetime(fracture_df["fecha_inicio_fractura"]).head()

0   2019-04-20
1   2018-11-02
2   2017-11-19
3   2017-11-21
4   2017-11-18
Name: fecha_inicio_fractura, dtype: datetime64[ns]

In [40]:
fecha_inicio = pd.to_datetime(fracture_df["fecha_inicio_fractura"])
(
    (fecha_inicio.dt.year == fracture_df["anio_if"])
    &
    (fecha_inicio.dt.month == fracture_df["mes_if"])
).value_counts()

True    4806
Name: count, dtype: int64

In [41]:
fecha_inicio.min(), fecha_inicio.max()

(Timestamp('2006-05-19 00:00:00'), Timestamp('2026-12-05 00:00:00'))

In [42]:
(
    fecha_inicio
    <=
    pd.to_datetime(fracture_df["fecha_fin_fractura"])
).value_counts()

True     4769
False      37
Name: count, dtype: int64

In [43]:
fracture_df.loc[
    pd.to_datetime(fracture_df["fecha_inicio_fractura"])
    >
    pd.to_datetime(fracture_df["fecha_fin_fractura"]),
    ["idpozo", "id_base_fractura_adjiv", "fecha_inicio_fractura", "fecha_fin_fractura"]
]

,idpozo,id_base_fractura_adjiv,fecha_inicio_fractura,fecha_fin_fractura
723,159764,899,2019-09-07,2018-09-10
880,135307,1214,2011-07-09,2010-10-22
906,143810,1240,2012-08-11,2012-08-01
1484,159864,1881,2018-08-23,2018-08-22
2279,161478,2846,2019-10-13,2019-09-13
2327,161201,2907,2015-06-29,2015-06-13
2490,132797,3098,2010-12-03,2010-02-10
2992,163246,3625,2022-05-08,2021-05-21
2993,163247,3626,2022-05-08,2021-05-21
3057,163058,3691,2022-02-25,2022-02-03


In [44]:
fechas = fracture_df.loc[
    pd.to_datetime(fracture_df["fecha_inicio_fractura"])
    >
    pd.to_datetime(fracture_df["fecha_fin_fractura"])
].copy()

fechas["dias_diferencia"] = (
    pd.to_datetime(fechas["fecha_inicio_fractura"])
    -
    pd.to_datetime(fechas["fecha_fin_fractura"])
).dt.days

fechas[["idpozo", "fecha_inicio_fractura", "fecha_fin_fractura", "dias_diferencia"]]

,idpozo,fecha_inicio_fractura,fecha_fin_fractura,dias_diferencia
723,159764,2019-09-07,2018-09-10,362
880,135307,2011-07-09,2010-10-22,260
906,143810,2012-08-11,2012-08-01,10
1484,159864,2018-08-23,2018-08-22,1
2279,161478,2019-10-13,2019-09-13,30
2327,161201,2015-06-29,2015-06-13,16
2490,132797,2010-12-03,2010-02-10,296
2992,163246,2022-05-08,2021-05-21,352
2993,163247,2022-05-08,2021-05-21,352
3057,163058,2022-02-25,2022-02-03,22


#### Findings

- The fecha_inicio_fractura column is currently stored as a string (object) but can be successfully converted to a valid datetime type without parsing errors.

- The year and month extracted from fecha_inicio_fractura match the values stored in anio_if and mes_if for 100% of the records, confirming that these attributes are fully derived from the complete date and therefore redundant.

- The temporal coverage ranges from May 2006 to December 2026, providing the earliest business event available across the analyzed datasets.

- A temporal consistency check comparing fecha_inicio_fractura and fecha_fin_fractura identified 37 records (0.77%) where the recorded start date occurs after the recorded end date.

- The observed inconsistencies follow repeated patterns across multiple wells and hydraulic fracturing records, suggesting that they are unlikely to be isolated typing mistakes. These records should therefore be considered potential source-data inconsistencies or differences in the semantic interpretation of the end date and should be reviewed before production deployment.

### Conclusion

The audit confirms that `fecha_inicio_fractura` is the primary business event date for hydraulic fracturing operations.

It should be used as the temporal key relating fact_fracturing to the shared dim_time, while anio_if and mes_if should be treated as derived attributes generated from the complete date.

### 9.1.3 Business Interpretation — `fecha_fin_fractura`

#### Objective

Evaluate whether `fecha_fin_fractura` represents a relevant business event for hydraulic fracturing operations and determine whether it should participate as an additional temporal relationship with the shared Time Dimension (dim_time).

In [45]:
fracture_df["fecha_fin_fractura"].dtype

dtype('O')

In [46]:
pd.to_datetime(fracture_df["fecha_fin_fractura"]).head()

0   2019-04-30
1   2018-11-03
2   2017-12-14
3   2017-12-15
4   2017-12-15
Name: fecha_fin_fractura, dtype: datetime64[ns]

In [47]:
fecha_fin = pd.to_datetime(fracture_df["fecha_fin_fractura"])

In [48]:
(
    fecha_fin.dt.year
    ==
    fracture_df["anio_ff"]
).value_counts()

True    4806
Name: count, dtype: int64

In [49]:
fecha_fin.min(), fecha_fin.max()

(Timestamp('2006-05-24 00:00:00'), Timestamp('2026-05-28 00:00:00'))

In [50]:
(
    fecha_fin
    >=
    pd.to_datetime(fracture_df["fecha_inicio_fractura"])
).value_counts()

True     4769
False      37
Name: count, dtype: int64

#### Findings

- The fecha_fin_fractura column is currently stored as a string (object) but can be successfully converted to a valid datetime type without parsing errors.

- The year extracted from fecha_fin_fractura matches the values stored in anio_ff for 100% of the records, confirming that anio_ff is a fully derived attribute and therefore redundant.

- The temporal coverage ranges from May 2006 to May 2026, representing the completion date of hydraulic fracturing operations across the available historical period.

- A chronological consistency check comparing fecha_fin_fractura with fecha_inicio_fractura shows that 4,769 records (99.23%) have a valid temporal sequence, while 37 records (0.77%) present an end date occurring before the recorded start date. These inconsistencies match those identified during the evaluation of fecha_inicio_fractura and should be documented as potential source-data quality issues.

- Unlike fecha_data, which represents an administrative loading date, fecha_fin_fractura describes a genuine business event corresponding to the completion of the hydraulic fracturing operation.

### Conclusion

The audit confirms that `fecha_fin_fractura` represents a valid business event and may serve as an additional temporal reference for hydraulic fracturing analyses.

Although fecha_inicio_fractura should remain the primary temporal key for fact_fracturing, fecha_fin_fractura may also be linked to the shared dim_time when analyses require evaluating the completion date of hydraulic fracturing operations.

The derived attribute anio_ff is functionally redundant and should not be stored independently in the dimensional model.

### 9.1.4 Business Interpretation — `fecha_data` (fracturing)

#### Objective

Evaluate whether `fecha_data` represents a business event or an administrative timestamp, and determine whether it should participate as a temporal key in the shared Time Dimension (dim_time).

In [51]:
fracture_df["fecha_data"].dtype

dtype('O')

In [52]:
pd.to_datetime(fracture_df["fecha_data"]).head()

0   2019-06-14 17:13:03.682790
1   2019-06-14 17:14:19.179874
2   2019-06-27 13:46:21.149350
3   2019-06-27 13:46:21.149350
4   2019-06-27 13:46:21.149350
Name: fecha_data, dtype: datetime64[ns]

In [53]:
fecha_data = pd.to_datetime(fracture_df["fecha_data"])

In [54]:
fecha_data.min(), fecha_data.max()

(Timestamp('2019-06-14 17:13:03.682790'),
 Timestamp('2026-07-02 11:25:04.014277'))

In [55]:
(
    fecha_data.dt.year
    ==
    fracture_df["anio_carga"]
).value_counts()

True    4806
Name: count, dtype: int64

In [56]:
(
    fecha_data.dt.month
    ==
    fracture_df["mes_carga"]
).value_counts()

True    4806
Name: count, dtype: int64

In [57]:
(
    fecha_data
    -
    pd.to_datetime(fracture_df["fecha_fin_fractura"])
).describe()

count                            4806
mean      877 days 11:58:25.816155568
std       976 days 23:25:28.291709328
min        -107 days +15:28:30.267206
25%           86 days 11:32:44.898731
50%          405 days 23:58:35.852504
75%      1509 days 02:20:31.345968256
max         5164 days 13:25:21.879779
dtype: object

In [58]:
(
    fecha_data
    <
    pd.to_datetime(fracture_df["fecha_fin_fractura"])
).value_counts()

False    4803
True        3
Name: count, dtype: int64

#### Findings

- The fecha_data column is stored as a string (object) but can be converted to a valid datetime type without inconsistencies.

- Unlike the production dataset, fecha_data contains timestamp information (date and time), indicating that it records the exact moment when the information was registered or loaded into the source system rather than the occurrence of the hydraulic fracturing operation itself.

- The year and month extracted from fecha_data match the values stored in anio_carga and mes_carga for 100% of the records, confirming that both attributes are fully derived from the complete timestamp and therefore redundant.

- The administrative timestamps span from June 2019 to July 2026, representing the period during which the source system was updated rather than the historical period of hydraulic fracturing activities.

- The elapsed time between fecha_fin_fractura and fecha_data varies considerably, ranging from 107 days before the recorded completion date to more than 5,100 days after. This wide variability demonstrates that fecha_data is not temporally tied to the hydraulic fracturing event itself and therefore should not be interpreted as a business event date.

- A consistency check between fecha_data and fecha_fin_fractura shows that 99.94% of the records were loaded on or after the completion date of the hydraulic fracturing operation. Only three records (0.06%) present an administrative timestamp preceding the recorded completion date, suggesting isolated source-data inconsistencies.

### Conclusion

The audit confirms that `fecha_data` represents an administrative load timestamp rather than a business event associated with hydraulic fracturing operations.

Consequently, this attribute should not be used as a temporal key for analytical relationships with dim_time. Its primary value lies in supporting data lineage, auditing, and ETL traceability.

The derived attributes anio_carga and mes_carga are functionally redundant and should not be stored independently in the dimensional model.

|Column|Decision|Reason|
|------|--------|------|
|fecha_data (production)|✅ fact_production → date_key|Monthly business reporting date.|
|fecha_inicio_fractura|✅ fact_fracturing → start_date_key|Business start date of the hydraulic fracturing operation.|
|fecha_fin_fractura|✅ fact_fracturing → end_date_key|Business completion date of the hydraulic fracturing operation.|
|fecha_data (fracturing)|❌ Exclude from dim_time|Administrative load timestamp used for auditing and ETL lineage.|
|anio, mes, anio_if, mes_if, anio_ff, anio_carga, mes_carga|❌ Remove|Fully derivable from the corresponding datetime attributes.|

## 9.2 dim_well Attribute Selection

#### Objective

The objective of this section is to determine which attributes from the production and fracturing datasets should be included in the well dimension (`dim_well`).

The selection process is based on data validation performed throughout this notebook, considering:
- stability at well level,
- temporal behavior,
- analytical value,
- and relationship with the business process.

### 9.2.1 `id_pozo`

|Column | Decision | Reason |
|-------|----------|--------|
|idpozo | ✅ Include	| Natural identifier of each well and shared key across production and fracturing datasets. |

### 9.2.2 Well Attribute Validation: `sigla`

#### Objective

Determine whether `sigla` is a stable attribute of each well and evaluate whether it should be included in the dim_well dimension.

In [59]:
production_sample.groupby("idpozo")["sigla"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: sigla, dtype: float64

#### Finding

Every well in the analyzed sample is associated with exactly one sigla value.

The descriptive statistics show:

- Minimum unique values per well: 1
- Maximum unique values per well: 1
- Mean unique values per well: 1.0

This indicates that sigla remains constant for each well.

In [60]:
production_sample.groupby("idpozo")["sigla"].nunique().max()

np.int64(1)

#### Finding

The maximum number of different sigla values associated with any well is 1, confirming that no well changes its identifier within the analyzed dataset.

In [61]:
production_sample.groupby("idpozo")["sigla"].nunique().sort_values(ascending=False).head(10)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
Name: sigla, dtype: int64

#### Finding

The inspected wells all have a single associated sigla, providing additional evidence that this attribute is stable across production records.

### Conclusion

`sigla` behaves as a stable descriptive attribute of a well.

Since each idpozo is associated with exactly one sigla, this column should be included in the dim_well dimension rather than in a fact table.

### 9.2.3 Well Attribute Validation: `tipoextraccion`

#### Objective

Determine whether `tipoextraccion` is a stable attribute of a well and evaluate whether it should be included in the dim_well dimension.

In [62]:
production_sample.groupby("idpozo")["tipoextraccion"].nunique().describe()

count    1534.000000
mean        1.231421
std         0.431056
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: tipoextraccion, dtype: float64

#### Finding

The analysis shows that most wells have a single extraction method, but not all of them.

- Mean unique extraction methods per well: 1.23
- Minimum: 1
- Maximum: 3

This indicates that some wells are associated with multiple extraction methods during their lifetime.

In [63]:
production_sample.groupby("idpozo")["tipoextraccion"].nunique().max()

np.int64(3)

#### Finding

The maximum number of different extraction methods observed for a single well is 3.

This demonstrates that the extraction method is not always constant.

In [64]:
production_sample.groupby("idpozo")["tipoextraccion"].nunique().sort_values(ascending=False).head(20)

idpozo
156606    3
131718    3
154793    3
153917    3
153918    3
153801    3
162196    2
131431    2
131600    2
161953    2
161954    2
161955    2
161688    2
161689    2
147301    2
146991    2
161720    2
161721    2
161722    2
161948    2
Name: tipoextraccion, dtype: int64

#### Finding

Several wells are associated with two or three different extraction methods, suggesting that this attribute may change during the operational life of a well.

In [65]:
pozo = (production_sample.groupby("idpozo")["tipoextraccion"].nunique())
pozo[pozo > 1]

idpozo
130297    2
130519    2
131068    2
131429    2
131431    2
         ..
162081    2
162082    2
162084    2
162085    2
162196    2
Name: tipoextraccion, Length: 349, dtype: int64

#### Finding

The dataset contains multiple wells with more than one extraction method, confirming that this behavior is not an isolated case.

In [66]:
production_sample.loc[
    production_sample["idpozo"] == 156606,
    ["anio", "mes", "tipoextraccion"]
].sort_values(["anio", "mes"])

,anio,mes,tipoextraccion
2264,2016,4,Sin Sistema de Extracción
3263,2016,5,Surgencia Natural
3561,2016,6,Surgencia Natural
4477,2016,7,Surgencia Natural
5528,2016,8,Surgencia Natural
5943,2016,9,Surgencia Natural
6559,2016,10,Surgencia Natural
7038,2016,11,Surgencia Natural
7637,2016,12,Surgencia Natural
614,2017,1,Surgencia Natural


### Conclusion

The inspection of individual wells confirms that `tipoextraccion` changes throughout the operational life of a well.

The observed transitions occur across different reporting periods rather than as inconsistent values within the same period, suggesting that the extraction method reflects the operational state of the well at a given time.

Therefore, tipoextraccion should not be considered a permanent well attribute and should not be stored in dim_well.

Instead, it should remain associated with the production records where the extraction method is reported for each month.

### 9.2.4 Well Attribute Validation: `tipoestado`

#### Objective

Determine whether `tipoestado` is a stable characteristic of each well or whether it changes over time.

If the well status changes during its operational life, the attribute should not be stored in dim_well.

In [67]:
production_sample.groupby("idpozo")["tipoestado"].nunique().describe()

count    1534.000000
mean        1.177314
std         0.424134
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         4.000000
Name: tipoestado, dtype: float64

#### Finding

Most wells are associated with a single operational status, although several wells present multiple statuses over time.

The maximum number of different statuses observed for a single well is four, suggesting that the operational status evolves throughout the life of a well.

In [68]:
production_sample.groupby("idpozo")["tipoestado"].nunique().max()

np.int64(4)

#### Finding

A single well can be associated with up to four different operational statuses, indicating that this attribute is not permanently associated with the well.

In [69]:
production_sample.groupby("idpozo")["tipoestado"].nunique().sort_values(ascending=False).head(20)

idpozo
131429    4
133430    4
158257    4
132772    3
131068    3
157695    3
136722    3
134528    3
132877    3
157715    3
155443    3
156472    3
137717    3
137600    3
156604    3
147463    3
147476    3
153801    3
156743    3
156804    3
Name: tipoestado, dtype: int64

#### Finding

Several wells present three or four different operational statuses, confirming that status changes are relatively common rather than isolated cases.

In [70]:
pozo = (production_sample.groupby("idpozo")["tipoestado"].nunique())
pozo[pozo > 1]

idpozo
115664    2
115728    2
126816    2
128093    2
129861    2
         ..
161512    2
161720    2
161952    2
161957    2
161958    2
Name: tipoestado, Length: 249, dtype: int64

#### Finding

The dataset contains numerous wells with more than one operational status, reinforcing that tipoestado varies over time.

In [71]:
production_sample.loc[
    production_sample["idpozo"] == 133430,
    ["anio", "mes", "tipoestado"]
].sort_values(["anio", "mes"])

,anio,mes,tipoestado
554,2015,1,En Estudio
960,2015,2,En Estudio
1565,2015,3,En Estudio
2642,2015,4,En Estudio
2887,2015,5,En Estudio
3974,2015,6,En Estudio
4614,2015,7,En Estudio
5473,2015,8,En Estudio
5807,2015,9,En Estudio
6206,2015,10,En Estudio


#### Finding

The inspection of an individual well shows a clear chronological evolution of its operational status:

- En Estudio
- Extracción Efectiva
- Parado Transitoriamente
- En Espera de Reparación

These transitions occur across consecutive reporting periods and represent different stages in the operational lifecycle of the well.

This confirms that tipoestado describes the monthly operational condition rather than an intrinsic property of the well itself.

### Conclusion

The analysis demonstrates that `tipoestado` changes throughout the operational life of a well.

Since the operational status is reported for each monthly production record and may evolve over time, it should not be stored in dim_well.

Instead, tipoestado should remain in fact_production, where it correctly represents the operational condition associated with each production period.

## 9.2.5 Well Attribute Validation: `profundidad`

#### Objective

Determine whether `profundidad` is a permanent characteristic of each well or whether it changes across production records.

If the reported depth remains constant for every well, it should be stored as an attribute in dim_well.

In [72]:
production_sample.groupby("idpozo")["profundidad"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: profundidad, dtype: float64

#### Finding

Every well is associated with a single depth value throughout the dataset.

The descriptive statistics show that all wells have exactly one unique value of profundidad, indicating that the reported depth is stable over time.

In [73]:
production_sample.groupby("idpozo")["profundidad"].nunique().max()

np.int64(1)

#### Finding

The maximum number of depth values observed for any well is one, confirming that depth does not vary across production records.

In [74]:
production_sample.groupby("idpozo")["profundidad"].nunique().sort_values(ascending=False).head(20)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
115733    1
115744    1
116022    1
124664    1
126736    1
126816    1
128093    1
128998    1
129050    1
129861    1
Name: profundidad, dtype: int64

#### Finding

The wells with the highest number of reported depth values still present only one unique value, providing additional evidence that depth remains constant for each well.

In [75]:
pozo = (production_sample.groupby("idpozo")["profundidad"].nunique())
pozo[pozo > 1]

Series([], Name: profundidad, dtype: int64)

#### Finding

Every well in the sample has a reported depth, and each well is associated with a single unique value.

No inconsistencies or changes in depth were identified.

In [76]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "profundidad"]
].sort_values(["anio", "mes"])

,anio,mes,profundidad
9069,2019,1,4230.0


#### Finding

The inspected well reports the same depth value throughout its production history, confirming the stability of this attribute at the individual well level.

### Conclusion

The validation confirms that `profundidad` is a stable physical characteristic of each well.

Since every well is associated with a single depth value across all production records, this attribute represents a permanent property rather than a time-dependent measurement.

Therefore, profundidad should be stored as an attribute of dim_well.

## 9.2.6 Well Attribute Validation: `tipopozo`

#### Objective

Determine whether `tipopozo` is a permanent characteristic of each well or whether it changes throughout the production history.

If each well is consistently associated with a single well type, this attribute should be stored in dim_well.

In [77]:
production_sample.groupby("idpozo")["tipopozo"].nunique().describe()

count    1534.000000
mean        1.071056
std         0.262029
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: tipopozo, dtype: float64

#### Findings

tipopozo is generally stable, but it is not constant for every well.

Most wells have a single reported well type, although some wells present two or even three different values throughout their production history.

In [78]:
production_sample.groupby("idpozo")["tipopozo"].nunique().max()

np.int64(3)

#### Findings

The maximum number of different tipopozo values observed for a single well is three, indicating that this attribute can change over time.

In [79]:
production_sample.groupby("idpozo")["tipopozo"].nunique().sort_values(ascending=False).head(20)

idpozo
131429    3
155443    3
137240    2
146989    2
147635    2
147316    2
145080    2
132772    2
132877    2
115664    2
136211    2
115728    2
137390    2
147476    2
136381    2
136402    2
133430    2
133844    2
136403    2
147463    2
Name: tipopozo, dtype: int64

#### Findings

Several wells were identified with multiple reported well types, confirming that this is not an isolated data quality issue.

In [80]:
pozo = (production_sample.groupby("idpozo")["tipopozo"].nunique())
pozo[pozo > 1]

idpozo
115664    2
115728    2
126816    2
128093    2
129861    2
         ..
159708    2
159712    2
159834    2
160154    2
161720    2
Name: tipopozo, Length: 107, dtype: int64

#### Findings

The list of affected wells confirms that multiple idpozo values are associated with more than one tipopozo.

Therefore, the variation occurs across the dataset and should be considered during the dimensional model design.

In [81]:
production_sample.loc[
    production_sample["idpozo"] == 126816,
    ["anio", "mes", "tipopozo"]
].sort_values(["anio", "mes"])

,anio,mes,tipopozo
174,2015,1,Petrolífero
1224,2015,2,Petrolífero
1457,2015,3,Petrolífero
2412,2015,4,Petrolífero
3285,2015,5,Petrolífero
3650,2015,6,Petrolífero
4422,2015,7,Petrolífero
5155,2015,8,Petrolífero
5591,2015,9,Petrolífero
6274,2015,10,Petrolífero


#### Findings

The temporal inspection shows a clear transition:

- During 2015 and most of 2016, the well is reported as Petrolífero.
- From October 2016 onward, it is reported as Gasífero.

This demonstrates that tipopozo changes over time rather than representing a permanent characteristic of the well.

### Conclusion

The analysis demonstrates that `tipopozo` is not a stable attribute of the well. Although most wells keep the same reported type, several wells change from one well type to another during their production history.

Since the attribute varies over time, storing it in dim_well would overwrite historical information and incorrectly assume that the well type is constant.

Therefore, tipopozo should remain associated with the production records (fact_production) rather than being stored as an attribute of dim_well.

### 9.2.7 Well Attribute Validation: `tipo_terminacion`

#### Objective

Determine whether `tipo_terminacion` is a stable characteristic of each well or whether it changes across hydraulic fracturing operations.

If every well is consistently associated with a single termination type, the attribute should be stored in dim_well.

In [82]:
fracture_df.groupby("idpozo")["tipo_terminacion"].nunique().describe()

count    4562.000000
mean        1.001315
std         0.036246
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
Name: tipo_terminacion, dtype: float64

#### Finding

Almost every well is associated with a single termination type.

However, a small number of wells present more than one reported termination type, indicating that the attribute is generally stable but not immutable.

In [83]:
fracture_df.groupby("idpozo")["tipo_terminacion"].nunique().max()

np.int64(2)

#### Finding

The maximum number of termination types observed for a single well is two.

In [84]:
fracture_df.groupby("idpozo")["tipo_terminacion"].nunique().sort_values(ascending=False).head(20)

idpozo
160161    2
153205    2
162224    2
162626    2
163058    2
163149    2
5993      1
5994      1
6002      1
6007      1
6013      1
6019      1
6031      1
6036      1
6040      1
6050      1
6051      1
6066      1
6068      1
6090      1
Name: tipo_terminacion, dtype: int64

#### Finding

Only six wells present multiple termination types, suggesting that these cases are uncommon but do exist.

In [85]:
pozo = (fracture_df.groupby("idpozo")["tipo_terminacion"].nunique())
pozo[pozo > 1]

idpozo
153205    2
160161    2
162224    2
162626    2
163058    2
163149    2
Name: tipo_terminacion, dtype: int64

#### Finding

The analysis identified six wells with more than one reported termination type.

Although the vast majority of wells remain constant, the presence of multiple values indicates that tipo_terminacion cannot be assumed to be permanently associated with a well.

In [86]:
fracture_df.loc[
    fracture_df["idpozo"] == 153205,
    [
        "id_base_fractura_adjiv",
        "fecha_inicio_fractura",
        "fecha_fin_fractura",
        "tipo_terminacion"
    ]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,fecha_fin_fractura,tipo_terminacion
1714,2111,2014-10-10,2014-10-15,Punzado
4482,5236,2023-08-27,2023-08-29,Camisas deslizables


#### Finding

The inspected well was fractured in two different periods:

- 2014: Punzado
- 2023: Camisas deslizables

This demonstrates that the termination type may change between fracturing operations performed on the same well.

In [87]:
fracture_df.loc[
    fracture_df["idpozo"].isin([153205, 160161, 162224, 162626, 163058, 163149]),
    [
        "idpozo",
        "fecha_inicio_fractura",
        "tipo_terminacion"
    ]
].sort_values(["idpozo", "fecha_inicio_fractura"])

,idpozo,fecha_inicio_fractura,tipo_terminacion
1714,153205,2014-10-10,Punzado
4482,153205,2023-08-27,Camisas deslizables
3050,160161,2022-02-07,Punzado
4493,160161,2022-02-07,Camisas deslizables
3040,162224,2021-03-09,Punzado
4497,162224,2023-05-07,Camisas deslizables
3082,162626,2021-03-09,Punzado
4504,162626,2022-11-06,Camisas deslizables
3057,163058,2022-02-25,Punzado
4511,163058,2022-02-25,Camisas deslizables


#### Finding

The temporal inspection of all affected wells reveals two distinct patterns:

- Some wells are associated with different termination types across separate fracturing campaigns conducted in different years.
- Other wells contain multiple termination types reported on the same fracturing date.

These observations indicate that tipo_terminacion is linked to individual fracturing records rather than representing a permanent characteristic of the well.

In [88]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    [
        "id_base_fractura_adjiv",
        "fecha_inicio_fractura",
        "fecha_fin_fractura",
        "tipo_terminacion"
    ]
]

,id_base_fractura_adjiv,fecha_inicio_fractura,fecha_fin_fractura,tipo_terminacion
3050,3684,2022-02-07,2022-02-09,Punzado
4493,5247,2022-02-07,2022-02-09,Camisas deslizables


#### Finding

The selected well contains two different fracturing records with the same start and end dates but different id_base_fractura_adjiv values and different termination types.

This demonstrates that multiple termination types can be recorded for the same fracturing campaign, confirming that tipo_terminacion describes the characteristics of an individual fracturing record rather than the well itself.

### Conclusion

The validation demonstrates that `tipo_terminacion` is not a permanent characteristic of a well.

Some wells are associated with different termination types across different hydraulic fracturing records, and in some cases multiple termination types are reported for the same fracturing campaign.

This indicates that the attribute describes the fracturing operation rather than the well itself.

Therefore, tipo_terminacion should remain in fact_fracturing, where it correctly represents the characteristics of each recorded fracturing event.

### 9.2.8 Well Attribute Validation: `proyecto`

#### Objective

Determine whether `proyecto` is a stable attribute of each well or whether it changes throughout the production history.

If every well is consistently associated with a single project, the attribute should be stored in dim_well. Otherwise, it should remain in fact_production.

In [89]:
production_sample.groupby("idpozo")["proyecto"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: proyecto, dtype: float64

#### Finding

Every well is associated with exactly one project.

The distribution shows that all wells have a single unique proyecto value, indicating that this attribute is completely stable across the production history.

In [90]:
production_sample.groupby("idpozo")["proyecto"].nunique().max()

np.int64(1)

#### Finding

The maximum number of different project values observed for a single well is one, confirming that no well changes project over time.

In [91]:
production_sample.groupby("idpozo")["proyecto"].nunique().sort_values(ascending=False).head(20)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
115733    1
115744    1
116022    1
124664    1
126736    1
126816    1
128093    1
128998    1
129050    1
129861    1
Name: proyecto, dtype: int64

#### Finding

The inspected wells are consistently associated with a single project value, providing additional evidence that proyecto does not vary within the same well.

In [92]:
pozo = (production_sample.groupby("idpozo")["proyecto"].nunique())
pozo[pozo > 1]

Series([], Name: proyecto, dtype: int64)

#### Finding

No wells were identified with more than one project.

This confirms that proyecto remains constant throughout the production history for every well in the dataset.

In [93]:
production_sample.loc[
    production_sample["idpozo"] == 72206,
    ["anio", "mes", "proyecto"]
].sort_values(["anio", "mes"])

,anio,mes,proyecto
9951,2019,1,GAS PLUS


#### Finding

The inspected well is consistently associated with the same project throughout the available production records.

This temporal inspection supports the previous findings and confirms that proyecto behaves as a stable well attribute.

### Conclusion

The analysis demonstrates that `proyecto` is a stable attribute of each well.

Every idpozo is consistently associated with a single project, and no changes were observed throughout the production history.

Therefore, proyecto should be stored as an attribute of dim_well.

### 9.2.9 Well Attribute Validation: `clasificacion`

#### Objective

Determine whether `clasificacion` is a stable attribute of each well or whether it changes throughout the production history.

If every well is consistently associated with a single classification, the attribute should be stored in dim_well. Otherwise, it should remain in fact_production.

In [94]:
production_sample.groupby("idpozo")["clasificacion"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: clasificacion, dtype: float64

#### Finding

Every well is associated with exactly one classification.

The distribution indicates that all wells have a single unique clasificacion value, suggesting that this attribute remains stable throughout the production history.

In [95]:
production_sample.groupby("idpozo")["clasificacion"].nunique().max()

np.int64(1)

#### Finding

The maximum number of different classification values observed for a single well is one.

This confirms that no well changes its classification over time.

In [96]:
production_sample.groupby("idpozo")["clasificacion"].nunique().sort_values(ascending=False).head(20)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
115733    1
115744    1
116022    1
124664    1
126736    1
126816    1
128093    1
128998    1
129050    1
129861    1
Name: clasificacion, dtype: int64

#### Finding

The inspected wells all have a single classification value, providing additional evidence that clasificacion is stable for every well.

In [97]:
pozo = (production_sample.groupby("idpozo")["clasificacion"].nunique())
pozo[pozo > 1]

Series([], Name: clasificacion, dtype: int64)

#### Finding

No wells were identified with more than one classification.

This confirms that clasificacion does not change during the production history of any well in the dataset.

In [98]:
production_sample.loc[
    production_sample["idpozo"] == 72206,
    ["anio", "mes", "clasificacion"]
].sort_values(["anio", "mes"])

,anio,mes,clasificacion
9951,2019,1,EXPLORACION


#### Finding

The inspected well maintains the same classification throughout the available production records.

This temporal inspection is consistent with the previous validations and supports the conclusion that clasificacion is a stable well attribute.

### Conclusion

The analysis demonstrates that `clasificacion` is a stable attribute of each well.

Every idpozo is consistently associated with a single classification, and no changes were observed throughout the production history.

Therefore, clasificacion should be stored as an attribute of dim_well.

### 9.2.10 Well Attribute Validation: `subclasificacion`

#### Objective

Determine whether `subclasificacion` is a stable attribute of each well or whether it changes throughout the production history.

If every well is consistently associated with a single sub-classification, the attribute should be stored in dim_well. Otherwise, it should remain in fact_production.

In [99]:
production_sample.groupby("idpozo")["subclasificacion"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: subclasificacion, dtype: float64

#### Finding

Every well is associated with exactly one sub-classification.

The distribution indicates that all wells have a single unique subclasificacion value, suggesting that this attribute remains stable throughout the production history.

In [100]:
production_sample.groupby("idpozo")["subclasificacion"].nunique().max()

np.int64(1)

#### Finding

The maximum number of different sub-classification values observed for a single well is one.

This confirms that no well changes its sub-classification over time.

In [101]:
production_sample.groupby("idpozo")["subclasificacion"].nunique().sort_values(ascending=False).head(20)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
115733    1
115744    1
116022    1
124664    1
126736    1
126816    1
128093    1
128998    1
129050    1
129861    1
Name: subclasificacion, dtype: int64

#### Finding

The inspected wells all present a single sub-classification value, providing additional evidence that subclasificacion is a stable attribute.

In [102]:
pozo = (production_sample.groupby("idpozo")["subclasificacion"].nunique())
pozo[pozo > 1]

Series([], Name: subclasificacion, dtype: int64)

#### Finding

No wells were identified with more than one sub-classification.

This confirms that subclasificacion remains constant throughout the production history for every well in the dataset.

In [103]:
production_sample.loc[
    production_sample["idpozo"] == 72206,
    ["anio", "mes", "subclasificacion"]
    ].sort_values(["anio", "mes"])

,anio,mes,subclasificacion
9951,2019,1,EXPLORATORIO PROFUNDO


#### Finding

The inspected well maintains the same sub-classification throughout the available production records.

This temporal inspection is consistent with the previous validations and supports the conclusion that subclasificacion is a stable well attribute.

### Conclusion

The analysis demonstrates that `subclasificacion` is a stable attribute of each well.

Every idpozo is consistently associated with a single sub-classification, and no changes were observed throughout the production history.

Therefore, subclasificacion should be stored as an attribute of dim_well.

### 9.2.11 Well Attribute Validation: `tipo_de_recurso`

#### Objective

Determine whether `tipo_de_recurso` remains constant for each well throughout the production history.

If every well is consistently associated with a single resource type, the attribute can be stored in a dimension table. Otherwise, it should remain in the fact table.

In [104]:
production_sample.groupby("idpozo")["tipo_de_recurso"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: tipo_de_recurso, dtype: float64

#### Finding

Every well is associated with exactly one resource type.

The distribution indicates that all wells have a single unique tipo_de_recurso value, suggesting that this attribute remains stable throughout the production history.

In [105]:
production_sample.groupby("idpozo")["tipo_de_recurso"].nunique().max()

np.int64(1)

#### Finding

The maximum number of different resource types observed for a single well is one.

No well changes its resource type over time.

In [106]:
production_sample.groupby("idpozo")["tipo_de_recurso"].nunique().sort_values(ascending=False).head(20)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
115733    1
115744    1
116022    1
124664    1
126736    1
126816    1
128093    1
128998    1
129050    1
129861    1
Name: tipo_de_recurso, dtype: int64

#### Finding

The inspected wells all have a single resource type, providing additional evidence that tipo_de_recurso is stable for every well.

In [107]:
pozo = (production_sample.groupby("idpozo")["tipo_de_recurso"].nunique())
pozo[pozo > 1]

Series([], Name: tipo_de_recurso, dtype: int64)

#### Finding

No wells were identified with more than one resource type.

This confirms that tipo_de_recurso remains constant throughout the production history.

In [108]:
production_sample.loc[
    production_sample["idpozo"] == 128093,
    ["anio", "mes", "tipo_de_recurso"]
].sort_values(["anio", "mes"])

,anio,mes,tipo_de_recurso
150,2015,1,NO CONVENCIONAL
1230,2015,2,NO CONVENCIONAL
1465,2015,3,NO CONVENCIONAL
2382,2015,4,NO CONVENCIONAL
2835,2015,5,NO CONVENCIONAL
3668,2015,6,NO CONVENCIONAL
4410,2015,7,NO CONVENCIONAL
5158,2015,8,NO CONVENCIONAL
5815,2015,9,NO CONVENCIONAL
6258,2015,10,NO CONVENCIONAL


#### Finding

The inspected well maintains the same resource type throughout all available production records.

This temporal inspection supports the conclusion that tipo_de_recurso is a stable attribute.

In [109]:
production_sample.groupby("formacion")["tipo_de_recurso"].nunique()

formacion
agrio                     1
basamento                 1
centenario                1
formación improductiva    1
grupo chubut              1
lajas                     1
los molles                1
lotena                    1
mulichinco                1
pozo d-129                1
precuyo                   1
punta rosada              1
quintuco                  1
sierras blancas           1
vaca muerta               1
Name: tipo_de_recurso, dtype: int64

#### Finding

Each geological formation is associated with a single resource type.

Within the available dataset, each geological formation is associated with a single resource type. This suggests that tipo_de_recurso is determined by the geological formation rather than by the individual well.

In [110]:
production_sample.groupby("tipo_de_recurso")["formacion"].nunique()

tipo_de_recurso
NO CONVENCIONAL    15
Name: formacion, dtype: int64

#### Finding

The only resource type present in the dataset (NO CONVENCIONAL) is associated with multiple geological formations.

This confirms that several geological formations belong to the same resource type, indicating a one-to-many relationship between tipo_de_recurso and formacion.

In [111]:
production_sample["tipo_de_recurso"].value_counts(dropna=False)

tipo_de_recurso
NO CONVENCIONAL    10000
Name: count, dtype: int64

#### Finding

All production records correspond to the same resource type (NO CONVENCIONAL).

Since the dataset contains a single category, tipo_de_recurso does not provide analytical discrimination and cannot be used to segment production data.

### Conclusion

Although `tipo_de_recurso` is a stable attribute and is functionally related to the geological formation, the source dataset contains only non-conventional wells. Consequently, the attribute has a single value across all records and does not provide additional analytical value.

For this reason, tipo_de_recurso will not be included in the dimensional model.

### 9.2.12 Well Attribute Validation: `rectificado`

#### Objective

Determine whether `rectificado` is a stable attribute at well level and evaluate whether it should be included in dim_well.

The analysis will verify if each well maintains the same rectificado value throughout the production records.

In [112]:
production_sample.groupby("idpozo")["rectificado"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: rectificado, dtype: float64

#### Finding

All wells have exactly one rectificado value across their production records.

This indicates that rectificado is a stable attribute at well level and does not change over time.

In [113]:
production_sample.groupby("idpozo")["rectificado"].nunique().max()

np.int64(1)

#### Finding

No well contains more than one rectificado value.

This confirms that there are no temporal changes or inconsistencies for this attribute.

In [114]:
production_sample.groupby("idpozo")["rectificado"].nunique().sort_values(ascending=False).head(20)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
115733    1
115744    1
116022    1
124664    1
126736    1
126816    1
128093    1
128998    1
129050    1
129861    1
Name: rectificado, dtype: int64

In [115]:
pozo = (production_sample.groupby("idpozo")["rectificado"].nunique())
pozo[pozo > 1]

Series([], Name: rectificado, dtype: int64)

#### Finding

No wells were identified with multiple rectificado values.

Therefore, rectificado behaves as a static well characteristic rather than a production event attribute.

In [116]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "rectificado"]
].sort_values(["anio", "mes"])

,anio,mes,rectificado
9069,2019,1,f


#### Finding

The inspected well maintains a single rectificado value during its available production history, consistent with the previous aggregation results.

In [117]:
production_sample["rectificado"].value_counts(dropna=False)

rectificado
f    10000
Name: count, dtype: int64

### Conclusion

`rectificado` is constant across the entire dataset, with all production records having the same value (f).

Although the attribute is stable at well level, it does not provide analytical value because it cannot be used to differentiate wells or production behavior.

### 9.2.13 Well Attribute Validation: `habilitado`

#### Objective

Determine whether `habilitado` is a stable well-level attribute and evaluate whether it provides analytical value for inclusion in dim_well.

The analysis will verify whether the attribute remains constant throughout the production records of each well and whether it contains enough variability to support well segmentation.

In [118]:
production_sample.groupby("idpozo")["habilitado"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: habilitado, dtype: float64

#### Finding

All wells contain exactly one habilitado value across their production records.

This indicates that habilitado is stable at well level and does not change during the available production history.

In [119]:
production_sample.groupby("idpozo")["habilitado"].nunique().max()

np.int64(1)

#### Finding

No well contains more than one habilitado value.

This confirms that the attribute does not show temporal variation within individual wells.

In [120]:
production_sample.groupby("idpozo")["habilitado"].nunique().sort_values(ascending=False).head(20)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
115733    1
115744    1
116022    1
124664    1
126736    1
126816    1
128093    1
128998    1
129050    1
129861    1
Name: habilitado, dtype: int64

In [121]:
pozo = (production_sample.groupby("idpozo")["habilitado"].nunique())
pozo[pozo > 1]

Series([], Name: habilitado, dtype: int64)

#### Finding

No wells were identified with more than one habilitado value.

This confirms that the attribute behaves as a static attribute rather than a time-dependent production variable.

In [122]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "habilitado"]
].sort_values(["anio", "mes"])

,anio,mes,habilitado
9069,2019,1,t


#### Finding

The inspected well maintains a single habilitado value, consistent with the previous aggregation results.

In [123]:
production_sample["habilitado"].value_counts(dropna=False)

habilitado
t    10000
Name: count, dtype: int64

### Conclusion

All production records have the same habilitado value (t).

Although the attribute is stable at well level, it does not provide analytical value because there is no variability across wells or production records.

### 9.2.14 Well Attribute Validation: `observaciones`

#### Objective

Determine whether `observaciones` contains relevant and stable information that should be included in dim_well.

The analysis will evaluate the presence of missing values, the amount of unique information available, and whether observations are associated with specific wells or production records.

In [124]:
production_sample["observaciones"].isna().sum()

np.int64(9983)

In [125]:
production_sample["observaciones"].notna().sum()

np.int64(17)

In [126]:
(production_sample["observaciones"].isna().mean() * 100)

np.float64(99.83)

#### Finding

observaciones is missing in 99.83% of the production records.

Only 17 records contain information, which indicates that this attribute has very limited coverage in the dataset.

In [127]:
production_sample.groupby("idpozo")["observaciones"].nunique()

idpozo
72206     0
72228     0
72232     0
75604     0
75605     1
         ..
166202    0
166203    0
166204    0
166205    0
166206    0
Name: observaciones, Length: 1534, dtype: int64

#### Finding

The attribute does not show meaningful variation at well level.

Most wells have no observations, and the few existing values correspond to automatically generated operational messages.

In [128]:
production_sample["observaciones"].value_counts(dropna=False).head(20)

observaciones
NaN                                               9983
Cargado automáticamente como [Sin movimientos]      17
Name: count, dtype: int64

#### Finding

The only non-null observation corresponds to an automatically generated message:

```text
Cargado automáticamente como [Sin movimientos]
```

The field does not contain descriptive information about the well characteristics.

In [129]:
production_sample.groupby("idpozo")["observaciones"].nunique().describe()

count    1534.000000
mean        0.005867
std         0.076396
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: observaciones, dtype: float64

#### Finding

The vast majority of wells have zero observations.

Only a very small number of wells contain this automatically generated message.

In [130]:
pozo = (production_sample.groupby("idpozo")["observaciones"].nunique())
pozo[pozo > 1]

Series([], Name: observaciones, dtype: int64)

#### Finding

No wells contain more than one observation value.

However, this stability is caused by the absence of information rather than by a meaningful well characteristic.

In [131]:
production_sample.loc[
    production_sample["idpozo"] == 145330,
    ["anio", "mes", "observaciones"]
].sort_values(["anio", "mes"])

,anio,mes,observaciones
182,2015,1,NaN
1060,2015,2,NaN
1617,2015,3,NaN
2574,2015,4,NaN
2833,2015,5,NaN
3881,2015,6,NaN
4359,2015,7,NaN
5428,2015,8,NaN
5755,2015,9,NaN
6156,2015,10,NaN


#### Finding

The observation appears only during periods where the system automatically records the status:
```text
Cargado automáticamente como [Sin movimientos]
```

This confirms that the field behaves as an operational annotation rather than a physical or descriptive well attribute.

### Conclusion

The analysis shows that `observaciones` does not represent a meaningful well-level attribute.

Although the field is technically stable for the few wells where values exist, the information coverage is extremely limited, with 99.83% of records missing. Additionally, the available values correspond only to automatically generated operational messages rather than descriptive characteristics of the wells.

Therefore, observaciones will not be included in dim_well because it does not provide analytical value for well segmentation, comparison, or business analysis.

### 9.2.15 Well Attribute Validation: `fechaingreso`

#### Objective

Determine whether fechaingreso represents a stable well-level attribute suitable for inclusion in dim_well.

The analysis evaluates data availability, uniqueness per well, temporal stability, and its potential analytical value for well lifecycle analysis.

In [132]:
production_sample["fechaingreso"].isna().sum()

np.int64(0)

In [133]:
production_sample["fechaingreso"].notna().sum()

np.int64(10000)

In [134]:
(production_sample["fechaingreso"].isna().mean() * 100)

np.float64(0.0)

#### Finding

fechaingreso contains information for all production records.

The attribute has complete coverage in the dataset.

In [135]:
production_sample["fechaingreso"].dtype

dtype('O')

In [136]:
production_sample["fechaingreso"].min(), production_sample["fechaingreso"].max()

('2015-02-09 11:22:06.741396', '2026-02-11 14:43:16.936002')

#### Finding

The column contains timestamp information ranging from 2015 to 2026.

However, the current data type is object, indicating that the field should be converted to datetime before further analysis.

In [137]:
production_sample.groupby("idpozo")["fechaingreso"].nunique().describe()

count    1534.000000
mean        6.518905
std        12.742749
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max        45.000000
Name: fechaingreso, dtype: float64

In [138]:
production_sample.groupby("idpozo")["fechaingreso"].nunique().max()

np.int64(45)

#### Finding

Unlike previous dim_well candidates, fechaingreso is not stable at well level.

Some wells contain multiple different dates throughout their production history.

In [139]:
pozo = (production_sample.groupby("idpozo")["fechaingreso"].nunique())
pozo[pozo > 1]

idpozo
115664     2
115728     2
126816    44
128093    44
129861    43
          ..
162193     2
162194     2
162195     2
162196     2
162200     2
Name: fechaingreso, Length: 905, dtype: int64

#### Finding

A significant number of wells have changing fechaingreso values.

Therefore, this attribute does not represent a fixed characteristic of the well.

In [140]:
production_sample.loc[
    production_sample["idpozo"] == 115664,
    ["anio", "mes", "fechaingreso"]
].sort_values(["anio", "mes"])

,anio,mes,fechaingreso
8931,2015,1,2015-02-09 11:22:06.741396
8292,2025,1,2025-02-17 17:39:10.815611


#### Finding

The same well presents different fechaingreso values depending on the production record.

This indicates that the field does not represent the original incorporation date of the well.

Instead, it appears to be related to data loading, system registration, or record update processes.

### Conclusion

The analysis confirms that `fechaingreso` should not be included in dim_well.

Although the field is complete and contains timestamp information, it does not behave as a well-level attribute because the same well can have multiple different values throughout the production history.

The changing behavior suggests that fechaingreso is related to data ingestion or administrative updates rather than the physical identity or characteristics of the well.

Therefore, the attribute is excluded from the dimensional model.

|Column|Decision|Reason|
|------|--------|-------------|
|idpozo|✅ dim_well| Natural identifier of the well.|
|sigla|✅ dim_well|Stable attribute (one value per well).|
|tipoextraccion|❌ fact_production|Changes over time.|
|tipoestado|❌ fact_production|Changes over time.|
|profundidad|✅ dim_well|Stable physical characteristic of the well.|
|tipopozo|❌ fact_production|Changes over time.|
|tipo_terminacion|❌ fact_production|Describes the fracturing record rather than the well.|
|proyecto|✅ dim_well|Stable descriptive attribute associated with the well.|
|clasificacion|✅ dim_weill|Stable well-level attribute.|
|subclasificacion|✅ dim_weill|Stable well-level attribute.|
|tipo_de_recurso|❌ Excluded|Constant value (NO CONVENCIONAL) across the entire dataset; provides no analytical value.|
|rectificado|❌ Excluded|Constant value (f) across the entire dataset; no analytical value.|
|habilitado|❌ Excluded|Constant value (f) across the entire dataset; no analytical value.|
|observaciones|❌ Excluded|Mostly null (99.83%) and contains only automatic operational messages, not descriptive well information.|
|fechaingreso|❌ Excluded|Changes over time and does not represent a stable well characteristic.|

## Conclusion

Based on the validation performed, `dim_well` will contain only attributes that represent stable characteristics of the well.

Time-dependent attributes such as extraction method, operational status, and fracturing characteristics will remain associated with their corresponding fact tables.

Attributes with no analytical value due to constant values, excessive missingness, or administrative behavior were excluded from the dimensional model.

## 9.3 dim_geology Attribute Selection

#### Objective

Determine which geological attributes should be included in dim_geology.

The analysis evaluates whether each candidate attribute represents a stable geological characteristic, its relationship with wells, and its analytical value for production and fracturing analysis.

### 9.3.1 Geology Attribute Validation: `formacion`

#### Objective

Determine whether `formacion` represents a stable geological characteristic that should be included in dim_geology.

The analysis evaluates whether each well is associated with a single geological formation, whether the attribute changes over time, and its potential analytical value for production and fracturing analysis.

In [141]:
production_sample.groupby("idpozo")["formacion"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: formacion, dtype: float64

#### Finding

Each well has exactly one associated geological formation.

The attribute does not show variation within the same well, indicating that formacion behaves as a stable geological characteristic.

In [142]:
production_sample.groupby("idpozo")["formacion"].nunique().max()

np.int64(1)

#### Finding

No well contains more than one formation.

This confirms that the attribute can be considered a well-level geological classification.

In [143]:
pozo = (production_sample.groupby("idpozo")["formacion"].nunique())
pozo[pozo > 1]

Series([], Name: formacion, dtype: int64)

#### Finding

No wells were identified with multiple formations.

The geological assignment remains consistent across all production records.

In [144]:
production_sample.groupby("idpozo")["formacion"].nunique().sort_values(ascending=False).head(20)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
115733    1
115744    1
116022    1
124664    1
126736    1
126816    1
128093    1
128998    1
129050    1
129861    1
Name: formacion, dtype: int64

In [145]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "formacion"]
].sort_values(["anio", "mes"])

,anio,mes,formacion
9069,2019,1,punta rosada


#### Finding

The inspected well maintains the same formation over its production history.

This behavior is consistent with a geological attribute rather than a time-dependent operational variable.

In [146]:
production_sample["formacion"].value_counts(dropna=False)

formacion
lajas                     4850
precuyo                   2661
vaca muerta               1483
basamento                  449
mulichinco                 179
quintuco                   124
formación improductiva     118
punta rosada                79
lotena                      27
los molles                  19
centenario                   4
sierras blancas              3
agrio                        2
grupo chubut                 1
pozo d-129                   1
Name: count, dtype: int64

#### Finding

The dataset contains multiple geological formations with different levels of representation.

The attribute provides meaningful segmentation capability for geological and production analysis.

### Conclusion

The analysis confirms that formacion represents a stable geological characteristic suitable for inclusion in dim_geology.

Each well is consistently associated with a single formation across all available production records. Additionally, the attribute contains multiple geological categories that can support comparative analysis of production performance across formations.

Therefore, formacion will be included in the dimensional model as part of dim_geology.

### 9.3.2 Geology Attribute Validation: `formprod`

#### Objective

Determine whether `formprod` represents a stable geological characteristic that should be included in dim_geology.

The analysis evaluates whether each well has a consistent productive formation over time, whether the attribute changes within the same well, and its analytical relevance for geological segmentation.

In [147]:
production_sample.groupby("idpozo")["formprod"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: formprod, dtype: float64

#### Finding

Each well has exactly one formprod value across all available production records.

This indicates that the productive formation behaves as a stable well-level geological attribute.

In [148]:
production_sample.groupby("idpozo")["formprod"].nunique().max()

np.int64(1)

#### Finding

No well contains more than one productive formation.

The attribute shows complete stability throughout the production history.

In [149]:
production_sample.groupby("idpozo")["formprod"].nunique().sort_values(ascending=False).head(20)

idpozo
72206     1
72228     1
72232     1
75604     1
75605     1
114888    1
114889    1
115664    1
115725    1
115728    1
115733    1
115744    1
116022    1
124664    1
126736    1
126816    1
128093    1
128998    1
129050    1
129861    1
Name: formprod, dtype: int64

In [150]:
pozo = (production_sample.groupby("idpozo")["formprod"].nunique())
pozo[pozo > 1]

Series([], Name: formprod, dtype: int64)

#### Finding

No wells were identified with multiple productive formations.

The assignment of formprod remains consistent across the dataset.

In [151]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "formprod"]
].sort_values(["anio", "mes"])

,anio,mes,formprod
9069,2019,1,PROS


#### Finding

The inspected well maintains a single productive formation value.

This behavior is consistent with a geological classification rather than a production event attribute.

In [152]:
production_sample["formprod"].value_counts(dropna=False)

formprod
LAJA    4850
PREC    2661
VMUT    1483
BASA     449
MULI     179
QTUC     124
FIMP     118
PROS      79
LOTE      27
LMOL      19
CENT       4
SBLA       3
AGRI       2
GCHU       1
POD1       1
Name: count, dtype: int64

#### Finding

formprod contains multiple productive formation categories represented through abbreviated codes.

The distribution matches the geological categories observed previously in formacion, suggesting that both columns represent the same underlying geological concept with different naming conventions.

In [153]:
production_sample[["formacion", "formprod"]].drop_duplicates().sort_values("formacion")

,formacion,formprod
8610,agrio,AGRI
0,basamento,BASA
8722,centenario,CENT
54,formación improductiva,FIMP
8549,grupo chubut,GCHU
4,lajas,LAJA
674,los molles,LMOL
650,lotena,LOTE
7725,mulichinco,MULI
9600,pozo d-129,POD1


#### Finding

Each geological formation is associated with exactly one productive formation code.

formprod behaves as a shortened coding system for formacion, rather than an independent geological classification.

In [154]:
production_sample.groupby("formacion")["formprod"].nunique()

formacion
agrio                     1
basamento                 1
centenario                1
formación improductiva    1
grupo chubut              1
lajas                     1
los molles                1
lotena                    1
mulichinco                1
pozo d-129                1
precuyo                   1
punta rosada              1
quintuco                  1
sierras blancas           1
vaca muerta               1
Name: formprod, dtype: int64

#### Finding

The relationship between formacion and formprod is one-to-one.

There are no cases where one formation is represented by multiple productive formation codes.

### Conclusion

The analysis confirms that `formprod` should not be included as an independent attribute in dim_geology.

Although the column is stable at well level, it contains the same geological information already represented by formacion, using abbreviated codes instead of full names.

To avoid redundancy and improve dimensional model clarity, dim_geology will keep formacion as the primary geological classification and exclude formprod.

### 9.3.3 Geology Attribute Validation: `tipo_de_recurso`

#### Objective

Determine whether `tipo_de_recurso` represents a meaningful geological classification suitable for inclusion in dim_geology.

The analysis evaluates:

- whether the resource type varies between wells,
- whether it provides analytical segmentation,
- and whether it contributes additional geological information beyond other attributes.

In [155]:
production_sample.groupby("idpozo")["tipo_de_recurso"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: tipo_de_recurso, dtype: float64

#### Finding

Each well has exactly one tipo_de_recurso value across all production records.

This indicates that the attribute is stable at well level and does not change throughout the production history.

In [156]:
production_sample.groupby("idpozo")["tipo_de_recurso"].nunique().max()

np.int64(1)

#### Finding

No well contains more than one resource type.

The attribute shows complete consistency across wells.

In [157]:
pozo = (production_sample.groupby("idpozo")["tipo_de_recurso"].nunique())
pozo[pozo > 1]

Series([], Name: tipo_de_recurso, dtype: int64)

#### Finding

No wells were identified with multiple tipo_de_recurso values.

The attribute remains constant for every well in the dataset.

In [158]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "tipo_de_recurso"]
].sort_values(["anio", "mes"])

,anio,mes,tipo_de_recurso
9069,2019,1,NO CONVENCIONAL


#### Finding

The inspected well maintains the same resource classification over time, confirming the stability observed in the previous validations.

In [159]:
production_sample["tipo_de_recurso"].value_counts(dropna=False)

tipo_de_recurso
NO CONVENCIONAL    10000
Name: count, dtype: int64

#### Finding

The complete dataset contains only one resource category: NO CONVENCIONAL.

Although the attribute is stable, it does not provide analytical differentiation because there is no variability between wells or production records.

### Conclusion

`tipo_de_recurso` represents a stable geological classification at well level, but it does not provide additional analytical value for the dimensional model because all records belong to the same category.

Therefore, the attribute will be excluded from dim_geology.

### 9.3.4 Geology Attribute Validation: `subtipo_recurso`

#### Objective

Determine whether `subtipo_recurso` represents a meaningful geological classification suitable for inclusion in dim_geology.

The analysis evaluates:

- whether the attribute is stable at well level,
- whether it changes throughout production history,
- and whether it provides geological segmentation beyond tipo_de_recurso.

In [160]:
production_sample.groupby("idpozo")["subtipo_recurso"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: subtipo_recurso, dtype: float64

#### Finding

Each well has exactly one subtipo_recurso value across all production records.

This indicates that the attribute is stable at well level and can be considered a geological characteristic of the well.

In [161]:
production_sample.groupby("idpozo")["subtipo_recurso"].nunique().max()

np.int64(1)

#### Finding

No well contains more than one resource subtype.

The attribute remains consistent throughout the available production history.

In [162]:
pozo = (production_sample.groupby("idpozo")["subtipo_recurso"].nunique())
pozo[pozo > 1]

Series([], Name: subtipo_recurso, dtype: int64)

#### Finding

No wells were identified with multiple subtipo_recurso values.

This confirms that the attribute does not behave as a time-dependent production variable.

In [163]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "subtipo_recurso"]
].sort_values(["anio", "mes"])

,anio,mes,subtipo_recurso
9069,2019,1,TIGHT


#### Finding

The inspected well maintains the same resource subtype over time, supporting the classification of this attribute as a stable geological property.

In [164]:
production_sample["subtipo_recurso"].value_counts(dropna=False)

subtipo_recurso
TIGHT    8469
SHALE    1531
Name: count, dtype: int64

#### Finding

Unlike tipo_de_recurso, subtipo_recurso provides meaningful geological segmentation.

The dataset contains two distinct categories:

- TIGHT
- SHALE

This attribute can therefore support comparative geological analysis.

### Conclusion

`subtipo_recurso` is a stable well-level geological attribute and provides analytical value by differentiating between resource subtypes.

Unlike tipo_de_recurso, which was constant across the dataset, subtipo_recurso introduces meaningful segmentation and should be included in the geological dimension.

### 9.3.5 Geology Attribute Validation: `tipo_reservorio`

#### Objective

Determine whether `tipo_reservorio` represents a stable geological characteristic of the well suitable for inclusion in dim_geology.

The analysis evaluates whether the reservoir type remains consistent for each well across different fracturing operations and whether it provides meaningful geological segmentation.

In [165]:
fracture_df.groupby("idpozo")["tipo_reservorio"].nunique().describe()

count    4562.000000
mean        0.993643
std         0.079485
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: tipo_reservorio, dtype: float64

#### Finding

For wells with available reservoir information, each well contains a single tipo_reservorio value across all fracturing records.

The presence of values with zero unique categories is explained by wells with missing reservoir information.

The attribute behaves as a stable geological characteristic at well level.

In [166]:
fracture_df.groupby("idpozo")["tipo_reservorio"].nunique().max()

np.int64(1)

#### Finding

No well contains more than one reservoir type across its fracturing records.

This confirms that tipo_reservorio does not change between fracturing operations for the same well.

In [167]:
pozo = (fracture_df.groupby("idpozo")["tipo_reservorio"].nunique())
pozo[pozo > 1]

Series([], Name: tipo_reservorio, dtype: int64)

#### Finding

No wells were identified with multiple tipo_reservorio values.

The reservoir classification remains consistent throughout the available fracturing history.

In [168]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "tipo_reservorio"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,tipo_reservorio
3050,3684,2022-02-07,CONVENCIONAL
4493,5247,2022-02-07,CONVENCIONAL


#### Finding

The inspected well shows the same reservoir classification across multiple fracturing records.

This supports the interpretation of tipo_reservorio as a geological property of the well rather than a fracturing event attribute.

In [169]:
fracture_df["tipo_reservorio"].value_counts(dropna=False)

tipo_reservorio
NO CONVENCIONAL    3837
CONVENCIONAL        933
NaN                  35
NO DISCRIMINADO       1
Name: count, dtype: int64

#### Finding

tipo_reservorio provides meaningful geological segmentation, with two main categories:

- NO CONVENCIONAL
- CONVENCIONAL

Missing values represent incomplete geological classification, while NO DISCRIMINADO appears as an isolated undefined category.

### Conclusion

`tipo_reservorio` represents a stable geological attribute associated with the well.

Although the attribute originates from fracture_df, its consistency across fracturing operations indicates that it should be integrated into dim_geology rather than fact_fracturing.

### 9.3.6 Geology Attribute Validation: `subtipo_reservorio`

#### Objective

Determine whether `subtipo_reservorio` represents a stable geological characteristic suitable for inclusion in dim_geology.

The analysis evaluates:

- whether the subtype remains consistent across fracturing records of the same well,
- whether it changes between different fracturing operations,
- and whether it provides meaningful geological segmentation.

In [170]:
fracture_df.groupby("idpozo")["subtipo_reservorio"].nunique().describe()

count    4562.000000
mean        0.824419
std         0.380505
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: subtipo_reservorio, dtype: float64

#### Finding

For wells with available subtype information, each well contains a single subtipo_reservorio value across all fracturing records.

The maximum value of one unique subtype per well indicates that this attribute does not vary between fracturing operations.

The lower mean value compared with tipo_reservorio is explained by missing subtype information in some wells.

In [171]:
fracture_df.groupby("idpozo")["subtipo_reservorio"].nunique().max()

np.int64(1)

#### Finding

No well contains more than one subtipo_reservorio classification.

This confirms that, when available, the attribute behaves as a stable geological characteristic.

In [172]:
pozo = (fracture_df.groupby("idpozo")["subtipo_reservorio"].nunique())
pozo[pozo > 1]

Series([], Name: subtipo_reservorio, dtype: int64)

#### Finding

No wells were identified with multiple reservoir subtypes.

The attribute does not show temporal variation across fracturing records.

In [173]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "subtipo_reservorio"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,subtipo_reservorio
3050,3684,2022-02-07,NaN
4493,5247,2022-02-07,NaN


#### Finding

The inspected well does not contain subtype information in its fracturing records.

This example highlights that subtipo_reservorio may suffer from incomplete data availability.

In [174]:
fracture_df["subtipo_reservorio"].value_counts(dropna=False)

subtipo_reservorio
SHALE    2933
NaN       974
TIGHT     899
Name: count, dtype: int64

#### Finding

subtipo_reservorio provides meaningful geological segmentation with two identified categories:

SHALE
TIGHT

Although 974 records have missing values, the available information shows that the attribute captures relevant geological differences within unconventional reservoirs.

The missing values represent incomplete geological classification rather than a lack of analytical relevance.

### Conclusion

`subtipo_reservorio` represents a stable geological characteristic associated with the well.

The attribute does not change between fracturing records for the same well and provides additional geological granularity beyond tipo_reservorio.

Despite the presence of missing values, the available classifications (SHALE and TIGHT) provide analytical value and should be incorporated into the geological dimension.

### 9.3.7 Geology Attribute Validation: `formacion_productiva`

#### Objective

Determine whether `formacion_productiva` represents a stable geological characteristic suitable for inclusion in dim_geology.

The analysis evaluates:

- whether the productive formation remains constant for each well,
- whether it changes over time,
- and whether it provides additional geological information compared with existing formation attributes.

In [175]:
fracture_df.groupby("idpozo")["formacion_productiva"].nunique().describe()

count    4562.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: formacion_productiva, dtype: float64

#### Finding

Each well contains exactly one formacion_productiva value across all available fracturing records.

This indicates that the productive formation behaves as a stable geological characteristic at well level.

In [176]:
fracture_df.groupby("idpozo")["formacion_productiva"].nunique().max()

np.int64(1)

#### Finding

No well contains more than one productive formation classification.

The attribute remains consistent across fracturing operations.

In [177]:
pozo = (fracture_df.groupby("idpozo")["formacion_productiva"].nunique())
pozo[pozo > 1]

Series([], Name: formacion_productiva, dtype: int64)

#### Finding

No wells were identified with multiple formacion_productiva values.

This confirms that the attribute does not behave as a fracturing-event variable.

In [178]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "formacion_productiva"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,formacion_productiva
3050,3684,2022-02-07,mina el carmen
4493,5247,2022-02-07,mina el carmen


#### Finding

The selected well maintains the same productive formation across multiple fracturing records.

This supports the interpretation of formacion_productiva as a geological well attribute.

In [179]:
fracture_df["formacion_productiva"].value_counts(dropna=False)

formacion_productiva
vaca muerta               2922
lajas                      338
mina el carmen             336
magallanes                 199
comodoro rivadavia         177
mulichinco                 159
huitrín                    151
los molles                 147
punta rosada                83
precuyo                     74
agrio                       68
cañadon seco                37
lotena                      19
el trebol                   13
bajo barreal                12
rayoso                      11
tordillo                    10
castillo                     8
quintuco                     8
springhill                   7
pozo d-129                   6
ANITA                        6
formación improductiva       5
chachao                      4
palermo aike                 2
loma montosa                 2
grupo neuquén                1
sierras blancas              1
Name: count, dtype: int64

#### Finding

formacion_productiva provides significant geological differentiation across fracturing records.

The attribute contains multiple geological formations, with vaca muerta being the dominant productive formation.

The variety of values indicates that this column provides additional geological granularity.

### Conclusion

`formacion_productiva` represents a stable geological attribute associated with the well.

Although the column originates from fracture_df, its consistency across fracturing operations indicates that it belongs in dim_geology rather than fact_fracturing.

|Column|Decision|Reason|
|------|--------|------|
|formacion|✅ dim_geology|Stable geological attribute associated with each well and useful for analytical segmentation.|
|formprdo|❌ Excluded|Redundant attribute. It represents a coded version of formacion and does not provide additional analytical value.|
|tipo_de_recurso|❌ Excluded|Constant value (NO CONVENCIONAL) across the entire dataset; provides no analytical differentiation.|
|subtipo_recurso|✅ dim_geology|Stable geological attribute with meaningful analytical differentiation (TIGHT vs SHALE).|
|tipo_reservorio|✅ dim_geology|Stable geological attribute across fracturing records with meaningful segmentation between conventional and unconventional reservoirs.|
|subtipo_reservorio|✅ dim_geology|Stable geological attribute with meaningful segmentation (SHALE vs TIGHT). Missing values represent incomplete classification.|
|formacion_productiva|✅ dim_geology|Stable geological attribute with meaningful differentiation between productive formations.|

## 9.4 dim_location Attribute Selection

### 9.4.1 Location Attribute Validation: `cuenca`

#### Objective

Determine whether cuenca represents a stable geographical attribute of the well suitable for inclusion in dim_location.

The analysis evaluates whether a well remains associated with the same basin throughout production records.

In [180]:
production_sample.groupby("idpozo")["cuenca"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: cuenca, dtype: float64

#### Finding

Each well is associated with a single basin across all available production records.

This indicates that cuenca behaves as a stable geographical characteristic of the well.

In [181]:
production_sample.groupby("idpozo")["cuenca"].nunique().max()

np.int64(1)

#### Finding

No well is associated with more than one basin.

The attribute does not show temporal variation throughout the production history.

In [182]:
pozo = (production_sample.groupby("idpozo")["cuenca"].nunique())
pozo[pozo > 1]

Series([], Name: cuenca, dtype: int64)

#### Finding

No wells were identified with multiple basin classifications.

This confirms that cuenca is a stable location attribute.

In [183]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "cuenca"]
].sort_values(["anio", "mes"])

,anio,mes,cuenca
9069,2019,1,NEUQUINA


#### Finding

The selected well maintains a consistent basin classification over time.

This supports the inclusion of cuenca as a well-level geographical attribute.

In [184]:
production_sample["cuenca"].value_counts(dropna=False)

cuenca
NEUQUINA           9998
GOLFO SAN JORGE       2
Name: count, dtype: int64

#### Finding

The dataset is overwhelmingly concentrated in the Neuquina basin.

Although the distribution is highly unbalanced, the attribute still provides geographical information and distinguishes wells located in different basins.

### Conclusion

`cuenca` represents a stable geographical attribute associated with each well.

Although the dataset is dominated by the Neuquina basin, the presence of multiple basin categories provides analytical value for geographic segmentation.

Therefore, cuenca should be included in dim_location.

### 9.4.2 Location Attribute Validation: `provincia`

#### Objective

Determine whether `provincia` represents a stable geographical attribute of the well suitable for inclusion in dim_location.

The analysis evaluates whether each well remains associated with the same province throughout production records and whether this attribute provides meaningful geographical segmentation.

In [185]:
production_sample.groupby("idpozo")["provincia"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: provincia, dtype: float64

#### Finding

Each well is associated with exactly one province across all available production records.

This indicates that provincia behaves as a stable geographical characteristic of the well and does not change over time.

In [186]:
production_sample.groupby("idpozo")["provincia"].nunique().max()

np.int64(1)

#### Finding

No well is associated with more than one province.

The attribute maintains a one-to-one relationship between the well and its geographical location.

In [187]:
pozo = (production_sample.groupby("idpozo")["provincia"].nunique())
pozo[pozo > 1]

Series([], Name: provincia, dtype: int64)

#### Finding

No wells were identified with multiple province classifications.

This confirms that provincia is not a temporal production attribute, but rather a stable location attribute.

In [188]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "provincia"]
].sort_values(["anio", "mes"])

,anio,mes,provincia
9069,2019,1,Neuquén


#### Finding

The selected well maintains the same province classification throughout its production history.

This supports the interpretation of provincia as a well-level geographical attribute.

In [189]:
production_sample["provincia"].value_counts(dropna=False)

provincia
Neuquén       5291
Rio Negro     4700
Mendoza          7
Chubut           1
Santa Cruz       1
Name: count, dtype: int64

#### Finding

The dataset contains wells distributed across five provinces, with most records concentrated in Neuquén and Río Negro.

Although the distribution is highly concentrated in two provinces, the attribute provides meaningful geographical segmentation for regional analysis.

### Conclusion

`provincia` represents a stable geographical attribute associated with each well.

The attribute remains constant throughout production records and provides analytical value by allowing production analysis by geographical region.

Therefore, provincia should be included in dim_location.

### 9.4.3 Location Attribute Validation: `coordenadax`

#### Objective

Determine whether `coordenadax` represents a stable spatial attribute of the well suitable for inclusion in dim_location.

The analysis evaluates whether the geographic longitude coordinate remains constant for each well throughout production records and whether it provides useful spatial information.

In [190]:
production_sample.groupby("idpozo")["coordenadax"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: coordenadax, dtype: float64

#### Finding

Each well has exactly one coordenadax value across all available production records.

This indicates that the longitude coordinate is a stable spatial characteristic of the well.

In [191]:
production_sample.groupby("idpozo")["coordenadax"].nunique().max()

np.int64(1)

#### Finding

No well presents more than one longitude coordinate.

The attribute does not vary over time and maintains a consistent spatial reference.

In [192]:
pozo = (production_sample.groupby("idpozo")["coordenadax"].nunique())
pozo[pozo > 1]

Series([], Name: coordenadax, dtype: int64)

#### Finding

No wells were identified with multiple longitude coordinates.

This confirms that coordenadax behaves as a fixed spatial attribute rather than a production-related variable.

In [193]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "coordenadax"]
].sort_values(["anio", "mes"])

,anio,mes,coordenadax
9069,2019,1,-68.248533


#### Finding

The selected well maintains a single longitude coordinate during its production history.

This supports the use of coordenadax as a well-level spatial attribute.

In [194]:
production_sample["coordenadax"].value_counts(dropna=False)

coordenadax
-69.181060    87
-69.255971    85
-69.227797    65
-69.223442    65
-69.232012    64
              ..
-68.716757     1
-68.661433     1
-68.602951     1
-68.798926     1
-68.607359     1
Name: count, Length: 1448, dtype: int64

#### Finding

The longitude values show spatial variability across wells, indicating that the attribute provides geographical discrimination.

Unlike categorical attributes such as province or basin, the coordinate allows more granular spatial analysis.

### Conclusion

`coordenadax` represents a stable spatial attribute associated with each well.

The coordinate remains constant over time and provides detailed geographical information that can support spatial analysis, mapping, and regional production studies.

Therefore, coordenadax should be included in dim_location.

### 9.4.4 Location Attribute Validation: `coordenaday`

#### Objective

Determine whether `coordenaday` represents a stable spatial attribute of the well suitable for inclusion in dim_location.

The analysis evaluates whether the latitude coordinate remains constant for each well throughout production records and whether it provides meaningful spatial information.

In [195]:
production_sample.groupby("idpozo")["coordenaday"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: coordenaday, dtype: float64

#### Finding

Each well has exactly one coordenaday value across all available production records.

This indicates that the latitude coordinate is a stable spatial characteristic of the well.

In [196]:
production_sample.groupby("idpozo")["coordenaday"].nunique().max()

np.int64(1)

#### Finding

No well presents more than one latitude coordinate.

The attribute remains constant over time and does not represent a changing production condition.

In [197]:
pozo = (production_sample.groupby("idpozo")["coordenaday"].nunique())
pozo[pozo > 1]

Series([], Name: coordenaday, dtype: int64)

#### Finding

No wells were identified with multiple latitude coordinates.

This confirms that coordenaday behaves as a fixed spatial attribute.

In [198]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "coordenaday"]
].sort_values(["anio", "mes"])

,anio,mes,coordenaday
9069,2019,1,-38.70794


#### Finding

The selected well maintains a single latitude coordinate during its production history.

This supports the interpretation of coordenaday as a well-level location attribute.

In [199]:
production_sample["coordenaday"].value_counts(dropna=False)

coordenaday
-39.079080    87
-38.860779    85
-38.863756    65
-38.863905    65
-38.862477    64
              ..
-38.312894     1
-38.196342     1
-38.322210     1
-38.196342     1
-38.322342     1
Name: count, Length: 1441, dtype: int64

#### Finding

The latitude values show spatial variability among wells.

This indicates that coordenaday provides additional spatial resolution beyond categorical attributes such as province or basin.

### Conclusion

`coordenaday` represents a stable spatial attribute associated with each well.

The coordinate remains unchanged throughout production records and provides precise geographic information useful for spatial analysis, visualization, and mapping.

Therefore, coordenaday should be included in dim_location.

|Column|Decision|Reason|
|------|--------|------|
|cuenca|✅ dim_location|Stable geographical attribute at well level.|
|provincia|✅ dim_location|Stable geographical attribute at well level, useful for regional segmentation.|
|coordenadax|✅ dim_location|Stable spatial coordinate providing detailed geographic information.|
|coordenaday|✅ dim_location|Stable spatial coordinate providing precise well location information.|

## 9.5 dim_field_area Attribute Selection

### 9.5.1 Area Attribute Validation: `idareapermisoconcesion`

#### Objective

Determine whether `idareapermisoconcesion` represents a stable field area identifier associated with each well and whether it is suitable for inclusion in dim_field_area.

The analysis evaluates whether each well remains linked to the same concession area throughout production records and whether this attribute provides meaningful segmentation of wells.

In [200]:
production_sample.groupby("idpozo")["idareapermisoconcesion"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: idareapermisoconcesion, dtype: float64

#### Finding

Each well is associated with exactly one idareapermisoconcesion value across all production records.

This indicates that the concession area identifier is a stable attribute of the well and does not change over time.

In [201]:
production_sample.groupby("idpozo")["idareapermisoconcesion"].nunique().max()

np.int64(1)

#### Finding

No well is associated with more than one concession area.

The relationship between the well and the concession identifier is one-to-one, supporting its use as a dimension attribute.

In [202]:
pozo = (production_sample.groupby("idpozo")["idareapermisoconcesion"].nunique())
pozo[pozo > 1]

Series([], Name: idareapermisoconcesion, dtype: int64)

#### Finding

No wells were identified with multiple concession area identifiers.

This confirms that idareapermisoconcesion represents a stable classification rather than a changing production characteristic.

In [203]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "idareapermisoconcesion"]
].sort_values(["anio", "mes"])

,anio,mes,idareapermisoconcesion
9069,2019,1,RGO


#### Finding

The selected well maintains the same concession area identifier during its available production history.

This supports the interpretation of idareapermisoconcesion as a fixed field area attribute.

In [204]:
production_sample["idareapermisoconcesion"].value_counts(dropna=False)

idareapermisoconcesion
FEO     4691
NDD     1163
SDD     1041
ANC      957
LCAM     940
LDL      321
RDM      194
LAC      150
ELOR     148
RGO      106
BNDS      63
X008      43
X009      43
LRIB      18
AGA       18
LAS1      16
CRAR       9
NRBN       8
LOCA       8
AGUH       7
OCT        7
CSN        6
BTN        6
PBN        5
DAD        4
LTII       4
BANE       3
CLMI       3
X205       2
LOAS       2
LMOL       2
FIM        2
LTI        2
LRII       1
VRG        1
SCA        1
LAS2       1
X002       1
PDYI       1
YAT        1
PAYO       1
Name: count, dtype: int64

#### Finding

The dataset contains multiple concession areas, with production records distributed among different field areas.

This attribute provides meaningful segmentation for comparing production performance across concession areas.

### Conclusion

`idareapermisoconcesion` represents a stable identifier of the concession area associated with each well.

The attribute remains constant throughout production records and provides analytical value for grouping wells according to their operational field area.

Therefore, idareapermisoconcesion should be included in dim_field_area.

|Column|Decision|Reason|
|------|--------|------|
|idareapermisoconcesion|✅ dim_field_area|Stable concession area identifier associated with each well and useful for field-level analysis.|

### 9.5.2 Area Attribute Validation: `areapermisoconcesion`

#### Objective

Determine whether `areapermisoconcesion` represents a stable descriptive attribute of the concession area associated with each well and whether it is suitable for inclusion in dim_field_area.

The analysis evaluates whether each well remains associated with the same concession area name throughout production records and whether this attribute provides meaningful operational segmentation.

In [205]:
production_sample.groupby("idpozo")["areapermisoconcesion"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: areapermisoconcesion, dtype: float64

#### Finding

Each well has exactly one areapermisoconcesion value throughout all production records.

This indicates that the concession area name is a stable attribute associated with the well.

In [206]:
production_sample.groupby("idpozo")["areapermisoconcesion"].nunique().max()

np.int64(1)

#### Finding

No well presents more than one concession area name.

The descriptive attribute maintains a consistent relationship with each well.

In [207]:
pozo = (production_sample.groupby("idpozo")["areapermisoconcesion"].nunique())
pozo[pozo > 1]

Series([], Name: areapermisoconcesion, dtype: int64)

#### Finding

No wells were identified with different concession area names over time.

This confirms that areapermisoconcesion is not a changing production attribute but a stable field classification.

In [208]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "areapermisoconcesion"]
].sort_values(["anio", "mes"])

,anio,mes,areapermisoconcesion
9069,2019,1,RIO NEUQUEN


#### Finding

The selected well maintains the same concession area description during its production history.

This supports the use of areapermisoconcesion as a descriptive attribute in dim_field_area.

In [209]:
production_sample["areapermisoconcesion"].value_counts(dropna=False)

areapermisoconcesion
ESTACION FERNANDEZ ORO           4691
AL NORTE DE LA DORSAL            1163
AL SUR DE LA DORSAL              1041
ANTICLINAL CAMPAMENTO             957
LOMA CAMPANA                      940
LOMA LA LATA - SIERRA BARROSA     321
RINCON DEL MANGRULLO              194
LA AMARGA CHICA                   150
EL OREJANO                        148
RIO NEUQUEN                       106
BANDURRIA SUR                      63
CORTADERA                          43
HUACALERA                          43
LA RIBERA BLOQUE I                 18
AGUADA DE LA ARENA                 18
LA ANGOSTURA SUR I                 16
CERRO ARENA                         9
NARAMBUENA                          8
LOS CALDENES                        8
AGUADA DEL CHAÑAR                   7
OCTOGONO                            7
CHIHUIDO DE LA SIERRA NEGRA         6
BAJO DEL TORO NORTE                 6
PASO DE LAS BARDAS NORTE            5
DADIN                               4
LAS TACANAS II               

#### Finding

The dataset contains multiple concession areas with different levels of representation.

This attribute provides meaningful operational segmentation and allows production analysis by concession area.

### Conclusion

`areapermisoconcesion` represents a stable descriptive attribute of the concession area associated with each well.

The attribute maintains a one-to-one relationship with each well, does not change over time, and provides meaningful information for grouping and comparing wells according to their operational area.

Therefore, areapermisoconcesion should be included in dim_field_area.

### 9.5.3 Area Attribute Validation: `idareayacimiento`

#### Objective

Determine whether `idareayacimiento` represents a stable identifier of the producing field/yacimiento associated with each well and whether it is suitable for inclusion in dim_field_area.

The analysis evaluates whether wells maintain the same yacimiento identifier throughout production records and whether this attribute provides meaningful segmentation for field-level analysis.

In [210]:
production_sample.groupby("idpozo")["idareayacimiento"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: idareayacimiento, dtype: float64

#### Finding

All wells have exactly one idareayacimiento value throughout the production records.

The absence of variation confirms that the yacimiento identifier is a stable characteristic associated with the well.

In [211]:
production_sample.groupby("idpozo")["idareayacimiento"].nunique().max()

np.int64(1)

#### Finding

The maximum number of different yacimiento identifiers associated with a single well is one.

This confirms that no well changes its assigned yacimiento during the available production history.

In [212]:
pozo = (production_sample.groupby("idpozo")["idareayacimiento"].nunique())
pozo[pozo > 1]

Series([], Name: idareayacimiento, dtype: int64)

#### Finding

No wells were identified with more than one idareayacimiento value.

This confirms that the yacimiento identifier is a stable attribute and does not change over time.

In [213]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "idareayacimiento"]
].sort_values(["anio", "mes"])

,anio,mes,idareayacimiento
9069,2019,1,RNQ


#### Finding

The selected well maintains a single yacimiento identifier during its available production history.

This supports the interpretation of idareayacimiento as a fixed field classification.

In [214]:
production_sample["idareayacimiento"].value_counts(dropna=False)

idareayacimiento
Z155    4691
GUA     1154
RQC     1041
LCLL     712
ACO      478
ANCS     391
Z116     257
LCAM     228
RDM      192
LAC      150
ELOR     148
RNQ      106
ANC       88
BDSS      63
Y310      43
Y325      43
Z051      31
Z349      20
AGA       18
LR        18
LAS1      16
BGO        9
Z077       9
Z259       8
LOCA       8
LCOL       7
Z262       7
LDL        6
BTN        6
CSN        6
PBN        5
DAD        4
LTII       4
ECOO       3
CLMI       3
BAEO       3
LMOL       2
FIM        2
LOAS       2
Z050       2
LOAM       2
RIPI       2
LTI        2
Z246       1
LRII       1
Z350       1
LAII       1
ETR        1
ETRI       1
CPAR       1
PDYI       1
Z157       1
PYOE       1
Name: count, dtype: int64

#### Finding

The dataset contains multiple yacimiento identifiers with different levels of representation.

This attribute provides meaningful segmentation for analyzing production performance at the field/yacimiento level.

### Conclusion

`idareayacimiento` represents a stable identifier of the producing field associated with each well.

The attribute maintains a consistent relationship with wells throughout production records and provides analytical value by allowing segmentation of wells according to their producing field.

### 9.5.4 Area Attribute Validation: `areayacimiento`

#### Objective

Determine whether `areayacimiento` represents a stable descriptive attribute of the producing field/yacimiento associated with each well and whether it is suitable for inclusion in dim_field_area.

The analysis evaluates whether each well remains associated with the same yacimiento name throughout production records and whether this attribute provides meaningful operational segmentation.

In [215]:
production_sample.groupby("idpozo")["areayacimiento"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: areayacimiento, dtype: float64

#### Finding

Each well has exactly one areayacimiento value throughout all production records.

This confirms that the yacimiento name is a stable descriptive attribute associated with the well.

In [216]:
production_sample.groupby("idpozo")["areayacimiento"].nunique().max()

np.int64(1)

#### Finding

The maximum number of different yacimiento names associated with a single well is one.

No well changes its assigned producing field name during the available production period.

In [217]:
pozo = (production_sample.groupby("idpozo")["areayacimiento"].nunique())
pozo[pozo > 1]

Series([], Name: areayacimiento, dtype: int64)

#### Finding

No wells were identified with multiple areayacimiento values.

The attribute behaves as a static classification rather than a variable that changes with production time.

In [218]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "areayacimiento"]
].sort_values(["anio", "mes"])

,anio,mes,areayacimiento
9069,2019,1,RIO NEUQUEN


#### Finding

The selected well maintains the same yacimiento name during its production record.

This supports the inclusion of areayacimiento as a descriptive field attribute.

In [219]:
production_sample["areayacimiento"].value_counts(dropna=False)

areayacimiento
ESTACION FERNANDEZ ORO         4691
GUANACO                        1154
RANQUIL CO                     1041
LOMA CAMPANA-LLL                712
ANTICLINAL CAMPAMENTO OESTE     478
ANTICLINAL CAMPAMENTO SUR       391
CUPEN MAHUIDA                   257
LOMA CAMPANA                    228
RINCON DEL MANGRULLO            192
LA AMARGA CHICA                 150
EL OREJANO                      148
RIO NEUQUEN                     106
ANTICLINAL CAMPAMENTO            88
BANDURRIA SUR                    63
CORTADERA                        43
HUACALERA                        43
BARROSA NORTE                    31
SIERRA BARROSA                   20
AGUADA DE LA ARENA               18
LA RIBERA BLOQUE I               18
LA ANGOSTURA SUR I               16
BARDA GONZALEZ                    9
CERRO ARENA                       9
NARAMBUENA                        8
LOS CALDENES                      8
LOMA COLORADA                     7
NORTE DEL OCTOGONO                7
LOMA LA LATA 

#### Finding

The dataset contains multiple yacimiento names with different production representation.

This attribute provides meaningful segmentation for grouping wells and analyzing production behavior by producing field.

### Conclusion

`areayacimiento` represents a stable descriptive attribute of the producing field associated with each well.

The attribute maintains a one-to-one relationship with wells, does not change over time, and provides analytical value for field-level segmentation.

Therefore, areayacimiento should be included in dim_field_area.

### 9.5.5 Area Attribute Validation: `yacimiento`

#### Objective

Determine whether `yacimiento` from fracture_df represents a stable field-level attribute associated with each well and whether it can be integrated into dim_field_area.

The analysis evaluates whether each well maintains the same yacimiento assignment across different fracturing records and whether this attribute provides meaningful segmentation for field-level analysis.

In [220]:
fracture_df.groupby("idpozo")["yacimiento"].nunique().describe()

count    4562.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: yacimiento, dtype: float64

#### Finding

Each well in the fracturing dataset is associated with exactly one yacimiento value.

This indicates that the yacimiento assignment remains stable across fracturing records.

In [221]:
fracture_df.groupby("idpozo")["yacimiento"].nunique().max()

np.int64(1)

#### Finding

The maximum number of different yacimiento values associated with a single well is one.

No well changes its assigned yacimiento between different fracturing operations.

In [222]:
pozo = (fracture_df.groupby("idpozo")["yacimiento"].nunique())
pozo[pozo > 1]

Series([], Name: yacimiento, dtype: int64)

#### Finding

No wells were identified with multiple yacimiento values.

The attribute behaves as a stable field classification rather than a variable associated with individual fracture events.

In [223]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "yacimiento"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,yacimiento
3050,3684,2022-02-07,EL TORDILLO
4493,5247,2022-02-07,EL TORDILLO


#### Finding

The selected well maintains the same yacimiento across different fracture records.

This confirms that the attribute describes the geological/field location of the well rather than the individual fracturing operation.

In [224]:
fracture_df["yacimiento"].value_counts(dropna=False)

yacimiento
LOMA CAMPANA-LLL            611
EL TORDILLO                 413
LA AMARGA CHICA             318
BANDURRIA SUR               212
LOMA CAMPANA                190
                           ... 
PASO DE LAS BARDAS NORTE      1
LA PAZ                        1
SUR PIEDRA CLAVADA            1
BAJO DEL TORO                 1
PARVA NEGRA                   1
Name: count, Length: 141, dtype: int64

#### Finding

The fracturing dataset contains multiple yacimiento values with different levels of representation.

This attribute provides meaningful segmentation for analyzing fracture operations across different producing fields.

### Conclusion

The `yacimiento` attribute from fracture_df represents a stable field-level characteristic associated with each well.

The attribute maintains a one-to-one relationship with idpozo, remains constant across multiple fracturing records, and provides analytical value for grouping wells according to their producing field.

Therefore, fracture_df["yacimiento"] should be incorporated into dim_field_area.

|Column|Decision|Reason|
|------|--------|------|
idareapermisoconcesion|✅ dim_field_area|Stable concession area identifier associated with each well and useful for field-level analysis.|
|areapermisoconcesion|✅ dim_field_area|Stable descriptive attribute representing the concession area associated with each well.|
|idareayacimiento|✅ dim_field_area|Stable identifier of the producing field/yacimiento associated with each well.|
|areayacimiento|✅ dim_field_area|Stable descriptive attribute representing the producing field/yacimiento associated with each well.|
|yacimiento|✅ dim_field_area|Stable field-level attribute associated with the producing yacimiento of each well.|

## 9.6 dim_company Attribute Selection

### 9.6.1 Company Attribute Validation: `empresa`

#### Objective

Determine whether `empresa` represents a stable company attribute associated with each well and whether it should be modeled as part of dim_company.

The analysis evaluates whether each well maintains the same operating company throughout production records or whether company ownership/operator information changes over time.

In [225]:
production_sample.groupby("idpozo")["empresa"].nunique().describe()

count    1534.000000
mean        1.107562
std         0.309927
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
Name: empresa, dtype: float64

#### Finding

Most wells are associated with a single company throughout their production history.

However, the maximum value of 2 indicates that some wells are associated with more than one company, suggesting that company information can change over time.

In [226]:
production_sample.groupby("idpozo")["empresa"].nunique().max()

np.int64(2)

#### Finding

Some wells have two different company values during their production records.

Therefore, empresa cannot be considered a static attribute of the well dimension (dim_well).

The attribute represents a time-dependent operational relationship.

In [227]:
pozo = (production_sample.groupby("idpozo")["empresa"].nunique())
pozo[pozo > 1]

idpozo
126816    2
128093    2
130021    2
130022    2
130158    2
         ..
159165    2
159302    2
159303    2
159473    2
159475    2
Name: empresa, Length: 165, dtype: int64

#### Finding

Multiple wells show changes in the associated company.

This indicates that company assignment is not permanently linked to the well and may reflect changes in operation, ownership, or reporting entity over time.

In [228]:
production_sample[["idempresa","empresa"]].drop_duplicates().sort_values("idempresa")

,idempresa,empresa
7721,YPF,YPF S.A.
0,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.


#### Finding

idempresa has a consistent relationship with the company name.

Each company identifier maps to a unique company description, making it suitable as the key attribute for dim_company.

In [229]:
production_sample["empresa"].value_counts(dropna=False)

empresa
YSUR ENERGÍA ARGENTINA S.R.L.    7721
YPF S.A.                         2279
Name: count, dtype: int64

#### Finding

The dataset contains two companies with different levels of production representation.

Both companies provide meaningful analytical segmentation for production analysis.

### Conclusion

`empresa` should be modeled as part of dim_company, but it should not be stored as a static attribute in dim_well.

The analysis shows that:

- The company identifier has a stable mapping with the company name.
- Some wells change associated company over time.
- Company information represents an operational relationship that belongs closer to production records than to the physical well entity.

### 9.6.2 Company Attribute Validation: `empresa_informante`

#### Objective

Determine whether `empresa_informante` represents a stable company attribute related to fracture operations and whether it can be integrated into the dim_company dimension.

The analysis evaluates:

- Whether a well can have multiple reporting companies.
- Whether the company information is stable within fracture records.
- Whether companies reported in fracture data match production companies.

In [230]:
fracture_df.groupby("idpozo")["empresa_informante"].nunique().describe()

count    4562.000000
mean        1.001096
std         0.033092
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
Name: empresa_informante, dtype: float64

#### Finding

Most wells are associated with a single empresa_informante.

However, the maximum value of 2 indicates that some wells have fracture records reported by more than one company.

This suggests that the attribute is not a permanent characteristic of the well.

In [231]:
fracture_df.groupby("idpozo")["empresa_informante"].nunique().max()

np.int64(2)

#### Finding

A single well can have up to two different reporting companies in fracture records.

Therefore, empresa_informante should not be stored inside dim_well.

It represents information associated with the fracture operation.

In [232]:
pozo = (fracture_df.groupby("idpozo")["empresa_informante"].nunique())
pozo[pozo > 1]

idpozo
68550     2
161998    2
161999    2
162000    2
162001    2
Name: empresa_informante, dtype: int64

#### Finding

Only a small number of wells present more than one company.

Even though this occurs infrequently, it confirms that company information belongs to the operational event rather than the physical well.

In [233]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "empresa_informante"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,empresa_informante
3050,3684,2022-02-07,TECPETROL S.A.
4493,5247,2022-02-07,TECPETROL S.A.


#### Finding

For individual wells, fracture records generally maintain the same reporting company.

This supports the interpretation that empresa_informante identifies the company associated with the fracture activity.

In [234]:
fracture_df["empresa_informante"].value_counts(dropna=False)

empresa_informante
YPF S.A.                                              2185
TECPETROL S.A.                                         770
COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.                  214
CHEVRON ARGENTINA S.R.L.                               211
CAPEX S.A.                                             187
SHELL ARGENTINA S.A.                                   166
TOTAL AUSTRAL S.A.                                     164
VISTA ENERGY ARGENTINA SAU                             159
PAMPA ENERGIA S.A.                                     130
PAN AMERICAN ENERGY SL                                 128
PLUSPETROL S.A.                                        113
VISTA OIL & GAS ARGENTINA SAU                           62
COMPAÑÍAS ASOCIADAS PETROLERAS S.A.                     58
PHOENIX GLOBAL RESOURCES S.A.                           55
OILSTONE ENERGIA S.A.                                   51
CGC ENERGIA SAU                                         48
PETROLERA EL TREBOL S.A.             

#### Finding

fracture_df contains a much broader set of companies than production_sample.

This indicates that the fracture dataset captures additional operators or reporting entities involved in hydraulic fracturing activities.

#### Objective

Evaluate the relationship between company attributes from production and fracture datasets in order to define the correct source for the dim_company dimension.

The analysis verifies:

- Whether production companies are also present in fracture operations.
- Which companies appear exclusively in fracture records.
- Whether the company dimension should be built from a single dataset or from both sources.

In [235]:
(
    set(production_sample["empresa"].dropna().unique())
    -
    set(fracture_df["empresa_informante"].dropna().unique())
)

{'YSUR ENERGÍA ARGENTINA S.R.L.'}

#### Finding

One company appears in production data but does not appear as an informant company in fracture records:

YSUR ENERGÍA ARGENTINA S.R.L.

This indicates that the fracture dataset does not contain a complete representation of all production companies.

In [236]:
(
    set(fracture_df["empresa_informante"].dropna().unique())
    -
    set(production_sample["empresa"].dropna().unique())
)

{'CAPEX S.A.',
 'CGC ENERGIA SAU',
 'CHEVRON ARGENTINA S.R.L.',
 'COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.',
 'COMPAÑÍAS ASOCIADAS PETROLERAS S.A.',
 'EXXONMOBIL EXPLORATION ARGENTINA S.R.L.',
 'GEOPARK ARGENTINA S.A.',
 'MEDANITO S.A.',
 'OILSTONE ENERGIA S.A.',
 'PAMPA ENERGIA S.A.',
 'PAN AMERICAN ENERGY SL',
 'PETROLERA EL TREBOL S.A.',
 'PHOENIX GLOBAL RESOURCES S.A.',
 'PLUSPETROL CUENCA NEUQUINA S.R.L.',
 'PLUSPETROL S.A.',
 'SHELL ARGENTINA S.A.',
 'SINOPEC ARGENTINA EXPLORATION AND PRODUCTION, INC.',
 'TECPETROL S.A.',
 'TOTAL AUSTRAL S.A.',
 'VISTA ENERGY ARGENTINA SAU',
 'VISTA OIL & GAS ARGENTINA SAU',
 'WINTERSHALL DEA ARGENTINA S.A'}

#### Finding

The fracture dataset contains a much broader set of companies than production data.

This confirms that:

fracture_df includes operators or companies involved in fracture activities that are not represented in the production sample.
The production dataset alone is insufficient to build the company dimension.

### Conclusion

Based on the analysis, the company dimension should be created by combining both datasets.

|Column|Decision|Reason|
|------|--------|------|
|idempresa|✅ dim_company|Stable identifier of the operating company.|
|empresa|✅ dim_company|Descriptive company attribute with one-to-one mapping to idempresa.|
|Source|✅ Combined dimension|Both datasets contain complementary company information.|

## 9.7 fact_production Attribute Selection

### 9.7.1 Production Attribute Validation: `prod_pet`

#### Objective

Evaluate the `prod_pet` attribute to determine whether it represents a valid production measure for the fact table.

The analysis verifies:

- Data completeness.
- Distribution of production values.
- Presence of zero-production periods.
- Temporal variability of production for each well.
- Suitability of the attribute as a quantitative measure in fact_production.

In [237]:
production_sample["prod_pet"].isna().sum()

np.int64(0)

#### Finding

The prod_pet attribute contains no missing values. Every production record has a valid oil production measurement.

In [238]:
production_sample["prod_pet"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, confirming complete coverage of oil production measurements across the dataset.

In [239]:
production_sample["prod_pet"].describe()

count    10000.000000
mean       139.053442
std        401.369898
min          0.000000
25%          0.000000
50%         12.554000
75%         87.187500
max       7983.260000
Name: prod_pet, dtype: float64

#### Finding

Oil production ranges from 0 to 7,983.26, with a median of 12.55 and a mean of 139.05.

The large difference between the median and the mean indicates a highly right-skewed distribution, where a relatively small number of wells produce substantially larger volumes than most others.

In [240]:
(production_sample["prod_pet"] == 0).sum()

np.int64(3264)

#### Finding

A total of 3,264 monthly records report zero oil production.

In [241]:
(production_sample["prod_pet"] == 0).mean() * 100

np.float64(32.64)

#### Finding

Approximately 32.64% of the monthly production records contain zero oil production.

These values likely represent inactive months, non-producing wells, or wells producing only other hydrocarbons, and therefore constitute valid business observations rather than missing data

In [242]:
production_sample.groupby("idpozo")["prod_pet"].nunique().describe()

count    1534.000000
mean        4.792047
std        10.193454
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        45.000000
Name: prod_pet, dtype: float64

#### Finding

Oil production varies over time for most wells.

Some wells maintain a constant production value throughout the sampled period, while others exhibit substantial variability, with up to 45 distinct production values.

This behavior is expected for a production measure recorded monthly.

In [243]:
production_sample.groupby("idpozo")["prod_pet"].nunique().max()

np.int64(45)

#### Finding

The maximum observed variability is 45 distinct production values for a single well, confirming that monthly production changes over time.

In [244]:
pozo = (production_sample.groupby("idpozo")["prod_pet"].nunique())
pozo[pozo > 1]

idpozo
115664     2
126816    43
128093    44
129861    37
129863     2
          ..
162193     2
162194     2
162195     2
162196     2
162200     2
Name: prod_pet, Length: 686, dtype: int64

#### Finding

A large number of wells exhibit multiple production values over time, confirming that prod_pet is a dynamic measure rather than a static well attribute.

In [245]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "prod_pet"]
].sort_values(["anio", "mes"])

,anio,mes,prod_pet
9069,2019,1,3.02


#### Finding

The selected well contains a production measurement for its reporting period, illustrating how prod_pet stores the monthly production associated with each well-time observation.

### Conclusion

`prod_pet` represents a quantitative business measure that changes over time and is fully populated across the dataset. The observed temporal variability, together with the absence of missing values, confirms that it should be modeled as a measure in the fact_production table.

Zero-production months are valid operational observations and should be preserved, as they represent meaningful production states rather than missing information.

### 9.7.2 Production Attribute Validation: `prod_gas`

#### Objective

Evaluate the `prod_gas` attribute to determine whether it represents a valid production measure for the fact table.

The analysis verifies:

- Data completeness.
- Distribution of production values.
- Presence of zero-production periods.
- Temporal variability of production for each well.
- Suitability of the attribute as a quantitative measure in fact_production.

In [246]:
production_sample["prod_gas"].isna().sum()

np.int64(0)

#### Finding

The prod_gas attribute contains no missing values. Every production record includes a valid natural gas production measurement.

In [247]:
production_sample["prod_gas"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, indicating complete coverage of gas production measurements throughout the dataset.

In [248]:
production_sample["prod_gas"].describe()

count    10000.000000
mean       474.500433
std        698.815557
min          0.000000
25%         51.872500
50%        230.574500
75%        646.878000
max      11667.250000
Name: prod_gas, dtype: float64

#### Finding

Gas production ranges from 0 to 11,667.25, with a median of 230.57 and a mean of 474.50.

As with oil production, the mean is substantially higher than the median, indicating a right-skewed distribution where a relatively small number of wells produce significantly larger gas volumes.

In [249]:
(production_sample["prod_gas"] == 0).sum()

np.int64(928)

#### Finding

A total of 928 monthly records report zero natural gas production.

In [250]:
(production_sample["prod_gas"] == 0).mean() * 100

np.float64(9.28)

#### Finding

Only 9.28% of the monthly records report zero gas production.

This percentage is considerably lower than that observed for oil production, suggesting that gas production is more consistently present across wells and reporting periods.

In [251]:
production_sample.groupby("idpozo")["prod_gas"].nunique().describe()

count    1534.000000
mean        6.183181
std        12.317170
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max        45.000000
Name: prod_gas, dtype: float64

#### Finding

Gas production varies over time for most wells.

The average number of distinct production values per well is 6.18, with some wells showing up to 45 different monthly production values, confirming the dynamic nature of gas production.

In [252]:
production_sample.groupby("idpozo")["prod_gas"].nunique().max()

np.int64(45)

#### Finding

The highest observed temporal variability is 45 distinct production values for a single well, demonstrating that gas production changes substantially over time.

In [253]:
pozo = (production_sample.groupby("idpozo")["prod_gas"].nunique())
pozo[pozo > 1]

idpozo
126816    43
128093    44
129861    37
129863    43
130021    44
          ..
162193     2
162194     2
162195     2
162196     2
162200     2
Name: prod_gas, Length: 842, dtype: int64

#### Finding

Many wells exhibit multiple gas production values throughout the observation period, confirming that prod_gas is a time-dependent business measure rather than a static well attribute.

In [254]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "prod_gas"]
].sort_values(["anio", "mes"])

,anio,mes,prod_gas
9069,2019,1,178.7


#### Finding

The selected well contains a valid monthly gas production measurement, illustrating how prod_gas captures production at the well-month level.

### Conclusion

`prod_gas` is a quantitative production measure that varies over time and contains no missing values. Its temporal variability and complete data coverage confirm that it should be modeled as a measure in the fact_production table.

Months with zero gas production represent valid operational conditions and should be retained as legitimate business observations.

### 9.7.3 Production Attribute Validation: `prod_agua`

#### Objective

Evaluate the `prod_agua` attribute to determine whether it represents a valid production measure for the fact table.

The analysis verifies:

- Data completeness.
- Distribution of production values.
- Presence of zero-production periods.
- Temporal variability of production for each well.
- Suitability of the attribute as a quantitative measure in fact_production.

In [255]:
production_sample["prod_agua"].isna().sum()

np.int64(0)

#### Finding

The prod_agua attribute contains no missing values. Every production record includes a valid water production measurement.

In [256]:
production_sample["prod_agua"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, confirming complete coverage of water production measurements throughout the dataset.

In [257]:
production_sample["prod_agua"].describe()

count    10000.000000
mean        65.897662
std        237.099770
min          0.000000
25%          1.160000
50%         13.275000
75%         43.260000
max       8369.990000
Name: prod_agua, dtype: float64

#### Finding

Water production ranges from 0 to 8,369.99, with a median of 13.28 and a mean of 65.90.

The mean is considerably higher than the median, indicating a right-skewed distribution where relatively few wells produce substantially larger water volumes.

In [258]:
(production_sample["prod_agua"] == 0).sum()

np.int64(2190)

#### Finding

A total of 2,190 monthly records report zero water production.

In [259]:
(production_sample["prod_agua"] == 0).mean() * 100

np.float64(21.9)

#### Finding

Approximately 21.90% of the monthly production records contain zero water production.

These observations represent valid operational conditions and should not be interpreted as missing data.

In [260]:
production_sample.groupby("idpozo")["prod_agua"].nunique().describe()

count    1534.000000
mean        5.513690
std        10.951986
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max        45.000000
Name: prod_agua, dtype: float64

#### Finding

Water production varies over time for most wells.

The average number of distinct production values per well is 5.51, with some wells exhibiting up to 45 different monthly production values, confirming the dynamic nature of the measure.

In [261]:
production_sample.groupby("idpozo")["prod_agua"].nunique().max()

np.int64(45)

#### Finding

The maximum observed temporal variability is 45 distinct production values for a single well, demonstrating substantial month-to-month variation in water production.

In [262]:
pozo = (production_sample.groupby("idpozo")["prod_agua"].nunique())
pozo[pozo > 1]

idpozo
115664     2
126816    43
128093    44
129861    37
129863    43
          ..
162193     2
162194     2
162195     2
162196     2
162200     2
Name: prod_agua, Length: 810, dtype: int64

#### Finding

Many wells present multiple water production values throughout the observation period, confirming that prod_agua is a dynamic business measure rather than a static well attribute.

In [263]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "prod_agua"]
].sort_values(["anio", "mes"])

,anio,mes,prod_agua
9069,2019,1,2.64


#### Finding

The selected well contains a valid monthly water production measurement, illustrating how prod_agua records production at the well-month level.

### Conclusion

`prod_agua` is a quantitative production measure that varies over time and contains no missing values. Its temporal variability and complete data coverage confirm that it should be modeled as a measure in the fact_production table.

Months with zero water production are valid business observations and should be preserved, as they represent actual operational conditions rather than missing information.

### 9.7.4 Production Attribute Validation: `iny_agua`

#### Objective

Evaluate the `iny_agua` attribute to determine whether it represents a valid injection measure for the fact table.

The analysis verifies:

- Data completeness.
- Distribution of injection values.
- Presence of zero-injection periods.
- Temporal variability of injection for each well.
- Suitability of the attribute as a quantitative measure in fact_production.

In [264]:
production_sample["iny_agua"].isna().sum()

np.int64(0)

#### Finding

The iny_agua attribute contains no missing values. Every monthly record includes a valid water injection measurement.

In [265]:
production_sample["iny_agua"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, indicating complete data coverage for water injection.

In [266]:
production_sample["iny_agua"].describe()

count    10000.000000
mean         8.253704
std        447.593245
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      29968.730000
Name: iny_agua, dtype: float64

#### Finding

Water injection ranges from 0 to 29,968.73, with a median of 0 and a mean of 8.25.

The distribution is extremely right-skewed, indicating that only a very small number of records contain water injection while the vast majority report no injection.

In [267]:
(production_sample["iny_agua"] == 0).sum()

np.int64(9996)

#### Finding

A total of 9,996 monthly records report zero water injection.

In [268]:
(production_sample["iny_agua"] == 0).mean() * 100

np.float64(99.96000000000001)

#### Finding

Approximately 99.96% of the monthly records contain zero water injection.

This suggests that water injection is performed only in a very small subset of wells and should be interpreted as a valid operational characteristic rather than missing information.

In [269]:
production_sample.groupby("idpozo")["iny_agua"].nunique().describe()

count    1534.000000
mean        1.001304
std         0.036096
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
Name: iny_agua, dtype: float64

#### Finding

Most wells have a single distinct injection value (typically zero).

The average number of distinct values per well is 1.00, indicating that only a few wells experience changes in water injection over time.

In [270]:
production_sample.groupby("idpozo")["iny_agua"].nunique().max()

np.int64(2)

#### Finding

The maximum observed temporal variability is 2 distinct values for a single well, reflecting that only a few wells transition between different injection levels.

In [271]:
pozo = (production_sample.groupby("idpozo")["iny_agua"].nunique())
pozo[pozo > 1]

idpozo
161900    2
161956    2
Name: iny_agua, dtype: int64

#### Finding

Only two wells exhibit more than one distinct water injection value, confirming that water injection activity is limited to a very small subset of wells.

In [272]:
production_sample.loc[
    production_sample["idpozo"] == 161900,
    ["anio", "mes", "iny_agua"]
].sort_values(["anio", "mes"])

,anio,mes,iny_agua
9886,2023,1,12695.29
9794,2026,1,29968.73


#### Finding

The selected well illustrates a valid injection history with different monthly water injection volumes, demonstrating that non-zero values represent real operational measurements.

### Conclusion

`iny_agua` is a quantitative operational measure representing monthly water injection volumes. Although nearly all records contain zero values, these zeros correspond to wells where no water injection occurred and therefore constitute valid business observations rather than missing data.

The attribute should be retained as a measure in the fact_production table because it captures an important operational process that applies to a subset of wells.

### 9.7.5 Production Attribute Validation: `iny_gas`

#### Objective

Evaluate the `iny_gas` attribute to determine whether it represents a valid injection measure for the fact table.

The analysis verifies:

- Data completeness.
- Distribution of injection values.
- Presence of zero-injection periods.
- Temporal variability of injection for each well.
- Suitability of the attribute as a quantitative measure in fact_production.

In [273]:
production_sample["iny_gas"].isna().sum()

np.int64(0)

#### Finding

The iny_gas attribute contains no missing values. Every monthly record includes a gas injection value.

In [274]:
production_sample["iny_gas"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, indicating complete data coverage.

In [275]:
production_sample["iny_gas"].describe()

count    10000.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: iny_gas, dtype: float64

#### Finding

All descriptive statistics are equal to 0, indicating that gas injection is never recorded in this dataset.

In [276]:
(production_sample["iny_gas"] == 0).sum()

np.int64(10000)

#### Finding

All 10,000 monthly records report zero gas injection.

In [277]:
(production_sample["iny_gas"] == 0).mean() * 100

np.float64(100.0)

#### Finding

100% of the observations contain a value of zero, confirming the absence of gas injection activity in the dataset.

In [278]:
production_sample.groupby("idpozo")["iny_gas"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: iny_gas, dtype: float64

#### Finding

Every well has exactly one distinct value for iny_gas, indicating that the attribute never changes over time.

In [279]:
production_sample.groupby("idpozo")["iny_gas"].nunique().max()

np.int64(1)

#### Finding

The maximum number of distinct values per well is 1, confirming the complete absence of temporal variability.

In [280]:
pozo = (production_sample.groupby("idpozo")["iny_gas"].nunique())
pozo[pozo > 1]

Series([], Name: iny_gas, dtype: int64)

#### Finding

No wells exhibit more than one gas injection value. The attribute remains constant for every well throughout the observation period.

In [281]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "iny_gas"]
].sort_values(["anio", "mes"])

,anio,mes,iny_gas
9069,2019,1,0.0


#### Finding

The selected well reports a gas injection value of 0 for the observed month, consistent with the behavior of the entire dataset.

### Conclusion

Although `iny_gas` contains no missing values, it provides no analytical information, since every record has the same value (0). The attribute exhibits neither variability across wells nor temporal variation.

Consequently, iny_gas should not be included in the dimensional model, as it does not contribute to analysis or reporting.

### 9.7.6 Production Attribute Validation: `iny_co2`

#### Objective

Evaluate the `iny_co2` attribute to determine whether it represents a valid injection measure for the fact table.

The analysis verifies:

- Data completeness.
- Distribution of injection values.
- Presence of zero-injection periods.
- Temporal variability of injection for each well.
- Suitability of the attribute as a quantitative measure in fact_production.

In [282]:
production_sample["iny_co2"].isna().sum()

np.int64(0)

#### Finding

The iny_co2 attribute contains no missing values. Every monthly production record includes a CO₂ injection value.

In [283]:
production_sample["iny_co2"].isna().mean() * 100

np.float64(0.0)

#### Finding

The iny_co2 attribute contains no missing values. Every monthly production record includes a CO₂ injection value.

In [284]:
production_sample["iny_co2"].describe()

count    10000.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: iny_co2, dtype: float64

#### Finding

All descriptive statistics are equal to 0, indicating that no CO₂ injection is recorded in the sampled production dataset.

In [285]:
(production_sample["iny_co2"] == 0).sum()

np.int64(10000)

#### Finding

All 10,000 monthly production records report zero CO₂ injection.

In [286]:
(production_sample["iny_co2"] == 0).mean() * 100

np.float64(100.0)

#### Finding

100% of the observations contain a value of zero, confirming the absence of CO₂ injection activity.

In [287]:
production_sample.groupby("idpozo")["iny_co2"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: iny_co2, dtype: float64

#### Finding

Each well has exactly one distinct value for iny_co2, indicating that the attribute remains constant throughout the observation period.

In [288]:
production_sample.groupby("idpozo")["iny_co2"].nunique().max()

np.int64(1)

#### Finding

The maximum number of distinct values per well is 1, confirming that no well exhibits temporal variation in CO₂ injection.

In [289]:
pozo = (production_sample.groupby("idpozo")["iny_co2"].nunique())
pozo[pozo > 1]

Series([], Name: iny_co2, dtype: int64)

#### Finding

No wells present more than one CO₂ injection value. The attribute is constant across all wells.

In [290]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "iny_co2"]
].sort_values(["anio", "mes"])

,anio,mes,iny_co2
9069,2019,1,0.0


#### Finding

The selected well reports a CO₂ injection value of 0, consistent with the behavior observed across the entire dataset.

### Conclusion

The `iny_co2` attribute is complete, but it contains a constant value (0) for every record. It shows neither temporal variation nor differences between wells, meaning it provides no analytical information for production analysis.

Therefore, iny_co2 should be excluded from the dimensional model because it does not contribute meaningful business information.

### 9.7.7 Production Attribute Validation: `iny_otro`

#### Objective

Evaluate the `iny_otro` attribute to determine whether it represents a valid injection measure for the fact table.

The analysis verifies:

- Data completeness.
- Distribution of injection values.
- Presence of zero-injection periods.
- Temporal variability of injection for each well.
- Suitability of the attribute as a quantitative measure in fact_production.

In [291]:
production_sample["iny_otro"].isna().sum()

np.int64(0)

#### Finding

The iny_otro attribute contains no missing values. Every production record includes a value for this measure.

In [292]:
production_sample["iny_otro"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, indicating complete data coverage.

In [293]:
production_sample["iny_otro"].describe()

count    10000.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: iny_otro, dtype: float64

#### Finding

All descriptive statistics are equal to 0, indicating that no injection classified as "other" is recorded in the sampled production dataset.

In [294]:
(production_sample["iny_otro"] == 0).sum()

np.int64(10000)

#### Finding

All 10,000 production records report an iny_otro value equal to zero.

In [295]:
(production_sample["iny_otro"] == 0).mean() * 100

np.float64(100.0)

#### Finding

100% of the observations contain a value of zero, suggesting that this variable does not capture any injection activity in the dataset.

In [296]:
production_sample.groupby("idpozo")["iny_otro"].nunique().describe()

count    1534.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: iny_otro, dtype: float64

#### Finding

Every well has exactly one distinct value for iny_otro, indicating that the attribute remains constant throughout the observation period.

In [297]:
production_sample.groupby("idpozo")["iny_otro"].nunique().max()

np.int64(1)

#### Finding

The maximum number of distinct values per well is 1, confirming that no well exhibits temporal variation in iny_otro.

In [298]:
pozo = (production_sample.groupby("idpozo")["iny_otro"].nunique())
pozo[pozo > 1]

Series([], Name: iny_otro, dtype: int64)

#### Finding

No wells present more than one iny_otro value. The attribute is constant across all wells.

In [299]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "iny_otro"]
].sort_values(["anio", "mes"])

,anio,mes,iny_otro
9069,2019,1,0.0


#### Finding

The selected well reports an iny_otro value of 0, consistent with the behavior observed across the entire dataset.

### Conclusion

The `iny_otro` attribute is complete but contains a constant value (0) for every production record. It exhibits no temporal variation and no differences between wells, indicating that it carries no analytical information.

Therefore, iny_otro should be excluded from the dimensional model because it does not contribute meaningful business information.

### 9.7.8 Production Attribute Validation: `tef`

#### Objective

Evaluate the `tef` attribute to determine whether it represents a meaningful production measure for the fact table.

The analysis verifies:

- Data completeness.
- Statistical distribution of values.
- Presence of zero values.
- Temporal variability for each well.
- Suitability of the attribute as a quantitative measure in fact_production.

In [300]:
production_sample["tef"].isna().sum()

np.int64(0)

#### Finding

The tef attribute contains no missing values. Every production record includes an effective operating time value.

In [301]:
production_sample["tef"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, indicating complete data coverage.

In [302]:
production_sample["tef"].describe()

count    10000.000000
mean        26.386154
std          9.491002
min          0.000000
25%         28.540000
50%         30.000000
75%         31.000000
max         31.000000
Name: tef, dtype: float64

#### Finding

The effective operating time ranges from 0 to 31, with a median of 30 and a mean of approximately 26.4. Most records correspond to wells operating during nearly the entire month, although some wells exhibit partial or no operating time.

In [303]:
(production_sample["tef"] == 0).sum()

np.int64(918)

#### Finding

A total of 918 records have a tef value equal to zero, indicating months in which the well had no effective operating time.

In [304]:
(production_sample["tef"] == 0).mean() * 100

np.float64(9.180000000000001)

#### Finding

Approximately 9.18% of the monthly records report zero effective operating time, suggesting that inactive months are relatively uncommon.

In [305]:
production_sample.groupby("idpozo")["tef"].nunique().describe()

count    1534.000000
mean        3.189700
std         4.662504
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        33.000000
Name: tef, dtype: float64

#### Finding

Most wells present multiple tef values over time, with an average of 3.19 distinct values per well. This indicates that effective operating time changes throughout the production history of each well.

In [306]:
production_sample.groupby("idpozo")["tef"].nunique().max()

np.int64(33)

#### Finding

The maximum number of distinct tef values observed for a single well is 33, confirming substantial temporal variability.

In [307]:
pozo = (production_sample.groupby("idpozo")["tef"].nunique())
pozo[pozo > 1]

idpozo
115664     2
126816    11
128093    14
129861    15
129863    10
          ..
162136     2
162193     2
162194     2
162195     2
162196     2
Name: tef, Length: 765, dtype: int64

#### Finding

A large number of wells exhibit more than one tef value, confirming that this measure varies naturally over time and captures monthly operational conditions.

In [308]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "tef"]
].sort_values(["anio", "mes"])

,anio,mes,tef
9069,2019,1,18.75


#### Finding

The selected well reports an effective operating time of 18.75, illustrating that wells may operate for only part of a month rather than the entire monthly period.

### Conclusion

The `tef` attribute is complete, exhibits meaningful variability across time, and reflects the monthly effective operating time of each well. Unlike the injection variables that remained constant, tef captures operational changes that directly influence production performance.

Therefore, tef should be retained as a quantitative measure in the fact_production table.

### 9.7.9 Production Attribute Validation: `vida_util`

#### Objective

Evaluate the `vida_util` attribute to determine whether it provides sufficient and meaningful information to be retained in the production fact table.

The analysis verifies:

- Data completeness.
- Percentage of missing values.
- Statistical distribution of available values.
- Temporal variability for each well.
- Analytical usefulness as a production measure.

In [309]:
production_sample["vida_util"].isna().sum()

np.int64(9983)

#### Finding

The vida_util attribute contains 9,983 missing values out of 10,000 records, indicating that the variable is almost entirely absent from the dataset.

In [310]:
production_sample["vida_util"].isna().mean() * 100

np.float64(99.83)

#### Finding

Approximately 99.83% of the records have missing values, making the attribute unsuitable for reliable analysis.

In [311]:
production_sample["vida_util"].describe()

count    17.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: vida_util, dtype: float64

#### Finding

Only 17 non-null values are available, and all of them are equal to zero. Consequently, the attribute exhibits no variability among its observed values.

In [312]:
(production_sample["vida_util"] == 0).sum()

np.int64(17)

#### Finding

All 17 available values are equal to 0, confirming the absence of meaningful information.

In [313]:
(production_sample["vida_util"] == 0).mean() * 100

np.float64(0.16999999999999998)

#### Finding

Only 0.17% of the dataset contains a recorded value, and every recorded value is zero.

In [314]:
production_sample.groupby("idpozo")["vida_util"].nunique().describe()

count    1534.000000
mean        0.005867
std         0.076396
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: vida_util, dtype: float64

#### Finding

Nearly all wells have no recorded value for vida_util. The maximum number of distinct values per well is 1, reflecting the absence of temporal variability.

In [315]:
production_sample.groupby("idpozo")["vida_util"].nunique().max()

np.int64(1)

#### Finding

The maximum number of distinct values per well is 1, indicating that the attribute never changes when present.

In [316]:
pozo = (production_sample.groupby("idpozo")["vida_util"].nunique())
pozo[pozo > 1]

Series([], Name: vida_util, dtype: int64)

#### Finding

No wells present more than one distinct vida_util value, confirming that the attribute does not vary over time.

In [317]:
production_sample.loc[
    production_sample["vida_util"].notna(),
    ["idpozo", "anio", "mes", "vida_util"]
].head(20)

,idpozo,anio,mes,vida_util
695,145330,2018,1,0.0
951,145330,2018,2,0.0
1898,145330,2018,3,0.0
2829,145330,2018,4,0.0
3440,145330,2018,5,0.0
4298,145330,2018,6,0.0
5049,145330,2018,7,0.0
7157,145330,2017,11,0.0
7716,145330,2017,12,0.0
9516,144931,2025,1,0.0


#### Finding

The few available observations all contain the value 0.0, providing no additional analytical information.

In [318]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "vida_util"]
].sort_values(["anio", "mes"])

,anio,mes,vida_util
9069,2019,1,NaN


#### Finding

The selected well has no recorded vida_util value, which is consistent with the overwhelming proportion of missing values across the dataset.

### Conclusion

The `vida_util` attribute has 99.83% missing values, and the remaining 0.17% of records contain only the value 0. It exhibits no meaningful variability and provides no useful analytical information for production analysis.

Therefore, vida_util should be excluded from the dimensional model.

### 9.7.10 Production Attribute Validation: `tipoextraccion`

#### Objective

Evaluate the `tipoextraccion` attribute to determine whether it represents a time-dependent operational status of the well or a stable well characteristic.

The analysis verifies:

- Data completeness.
- Number of categories.
- Temporal consistency for each well.
- Whether the attribute changes throughout the production history.
- Suitability for inclusion in the fact_production table.

In [319]:
production_sample["tipoextraccion"].isna().sum()

np.int64(0)

#### Finding

The tipoextraccion attribute contains no missing values. Every production record specifies the extraction method used during that month.

In [320]:
production_sample["tipoextraccion"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, indicating complete data coverage.

In [321]:
production_sample["tipoextraccion"].value_counts(dropna=False)

tipoextraccion
Surgencia Natural            6620
Plunger Lift                 2317
Bombeo Mecánico               758
Sin Sistema de Extracción     286
Gas Lift                       16
Otros Tipos de Extracción       3
Name: count, dtype: int64

#### Finding

The dataset contains six extraction methods. The most common is Surgencia Natural, followed by Plunger Lift and Bombeo Mecánico. The remaining methods are relatively uncommon but represent valid operational states.

In [322]:
production_sample.groupby("idpozo")["tipoextraccion"].nunique().describe()

count    1534.000000
mean        1.231421
std         0.431056
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: tipoextraccion, dtype: float64

#### Finding

Most wells have a single extraction method throughout the observed period, but some wells present changes over time. On average, wells have 1.23 distinct extraction methods, indicating that operational transitions occur in a subset of wells.

In [323]:
production_sample.groupby("idpozo")["tipoextraccion"].nunique().max()

np.int64(3)

#### Finding

The maximum number of distinct extraction methods observed for a single well is 3, confirming that extraction systems may change during the production life of a well.

In [324]:
pozo = (production_sample.groupby("idpozo")["tipoextraccion"].nunique())
pozo[pozo > 1]

idpozo
130297    2
130519    2
131068    2
131429    2
131431    2
         ..
162081    2
162082    2
162084    2
162085    2
162196    2
Name: tipoextraccion, Length: 349, dtype: int64

#### Finding

Several wells exhibit more than one extraction method, demonstrating that this attribute is time-dependent rather than a permanent characteristic of the well.

In [325]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "tipoextraccion"]
].sort_values(["anio", "mes"])

,anio,mes,tipoextraccion
9069,2019,1,Surgencia Natural


#### Finding

The selected well uses Surgencia Natural during the observed month. While this particular example does not show a transition, the previous analysis confirms that other wells do experience changes in extraction method over time. 

### Conclusion

The `tipoextraccion` attribute is complete, categorical, and represents the extraction system used during each production period. Although many wells retain the same extraction method, others change it over time, indicating that it reflects the operational status of the well rather than a permanent well characteristic.

Therefore, tipoextraccion should be retained as a descriptive operational attribute in the fact_production table.

### 9.7.11 Production Attribute Validation: `tipoestado`

#### Objective

Evaluate the `tipoestado` attribute to determine whether it represents a time-dependent operational status of the well or a stable well characteristic.

The analysis verifies:

- Data completeness.
- Distribution of operational states.
- Temporal consistency for each well.
- Whether the operational status changes throughout the production history.
- Suitability for inclusion in the fact_production table.

In [326]:
production_sample["tipoestado"].isna().sum()

np.int64(0)

#### Finding

The tipoestado attribute contains no missing values. Every production record has an associated operational status.

In [327]:
production_sample["tipoestado"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, indicating complete data availability for all production records.

In [328]:
production_sample["tipoestado"].value_counts(dropna=False)

tipoestado
Extracción Efectiva                   9196
Parado Transitoriamente                288
En Estudio                             278
En Espera de Reparación                 50
Abandonado                              43
A Abandonar                             34
Otras Situación Inactivo                28
En Reserva de Gas                       26
En Reserva para Recup. Sec./Asist.      20
Abandono Temporario                     17
En Reparación                           11
En Inyección Efectiva                    4
Parado Alta Relación Agua/Petróleo       3
Otras Situación Activo                   1
Parado Alta Relación Gas/Petróleo        1
Name: count, dtype: int64

#### Finding

The dataset contains multiple operational states. Extracción Efectiva is by far the dominant category, representing the normal operating condition of most production records. The remaining categories correspond to inactive, maintenance, abandonment, or special operational situations.

In [329]:
production_sample.groupby("idpozo")["tipoestado"].nunique().describe()

count    1534.000000
mean        1.177314
std         0.424134
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         4.000000
Name: tipoestado, dtype: float64

#### Finding

Most wells maintain a single operational state during the observed period. However, the average of 1.18 distinct states per well indicates that some wells experience operational status changes throughout their production history.

In [330]:
production_sample.groupby("idpozo")["tipoestado"].nunique().max()

np.int64(4)

#### Finding

A single well can present up to 4 different operational states, confirming that this attribute may change over time.

In [331]:
pozo = (production_sample.groupby("idpozo")["tipoestado"].nunique())
pozo[pozo > 1]

idpozo
115664    2
115728    2
126816    2
128093    2
129861    2
         ..
161512    2
161720    2
161952    2
161957    2
161958    2
Name: tipoestado, Length: 249, dtype: int64

#### Finding

Several wells exhibit more than one operational state, demonstrating that tipoestado is a dynamic attribute that reflects changes in well operation over time.

In [332]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "tipoestado"]
].sort_values(["anio", "mes"])

,anio,mes,tipoestado
9069,2019,1,Extracción Efectiva


#### Finding

The selected well is classified as Extracción Efectiva during the observed month. Although this example does not show a status transition, the previous analysis confirms that other wells undergo changes in operational status throughout their production history.

### Conclusion

The `tipoestado` attribute is complete, categorical, and represents the operational condition of each well during every production period. Since multiple wells change operational status over time, the attribute is time-dependent and should not be modeled as a static well characteristic.

Therefore, tipoestado should be retained as a descriptive operational attribute in the fact_production table.

### 9.7.12 Production Attribute Validation: `tipopozo`

#### Objective

Evaluate the `tipopozo` attribute to determine whether it represents a time-dependent operational classification of the well or a stable well characteristic.

The analysis verifies:

- Data completeness.
- Distribution of well types.
- Temporal consistency for each well.
- Whether the well type changes throughout the production history.
- Suitability for inclusion in the fact_production table.

In [333]:
production_sample["tipopozo"].isna().sum()

np.int64(0)

#### Finding

The tipopozo attribute contains no missing values. Every production record includes a well type classification.

In [334]:
production_sample["tipopozo"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, indicating complete data coverage.

In [335]:
production_sample["tipopozo"].value_counts(dropna=False)

tipopozo
Gasífero       8338
Petrolífero    1253
Otro tipo       405
Sumidero          4
Name: count, dtype: int64

#### Finding

The dataset contains four well type categories. Gasífero is the dominant category, followed by Petrolífero, while Otro tipo and Sumidero appear only in a small number of records.

In [336]:
production_sample.groupby("idpozo")["tipopozo"].nunique().describe()

count    1534.000000
mean        1.071056
std         0.262029
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: tipopozo, dtype: float64

#### Finding

Most wells maintain a single well type throughout the observed period. However, an average of 1.07 distinct values per well indicates that some wells change their operational classification over time.

In [337]:
production_sample.groupby("idpozo")["tipopozo"].nunique().max()

np.int64(3)

#### Finding

A well can present up to 3 different well type classifications, confirming that this attribute is not always static.

In [338]:
pozo = (production_sample.groupby("idpozo")["tipopozo"].nunique())
pozo[pozo > 1]

idpozo
115664    2
115728    2
126816    2
128093    2
129861    2
         ..
159708    2
159712    2
159834    2
160154    2
161720    2
Name: tipopozo, Length: 107, dtype: int64

#### Finding

Several wells present more than one well type during their production history, demonstrating that tipopozo may change over time and should not be considered a permanent well characteristic.

In [339]:
production_sample.loc[
    production_sample["idpozo"] == 72228,
    ["anio", "mes", "tipopozo"]
].sort_values(["anio", "mes"])

,anio,mes,tipopozo
9069,2019,1,Gasífero


#### Finding

The selected well is classified as Gasífero during the observed period. Although this example shows a single classification, the previous analysis indicates that other wells experience changes in well type over time.

### Conclusion

The `tipopozo` attribute is complete, categorical, and describes the operational classification of a well during each production period. Although most wells retain the same classification, some wells change well type over time, demonstrating that the attribute is time-dependent rather than a permanent well characteristic.

Therefore, tipopozo should be retained as a descriptive operational attribute in the fact_production table.

|Column|Decision|Reason|
|------|--------|------|
|prod_pet|✅ fact_production|Quantitative oil production measure recorded monthly. It varies over time for each well, contains no missing values, and represents a core business metric for production analysis.|
|prod_gas|✅ fact_production|Quantitative monthly natural gas production measure. It varies over time for each well, contains no missing values, and represents a core business metric for production analysis.|
|prod_agua|✅ fact_production|Quantitative monthly water production measure. It varies over time for each well, contains no missing values, and represents an essential business metric for production analysis.|
|iny_agua|✅ fact_production|Quantitative monthly water injection measure. Although most records contain zero values, these represent valid operational observations and the attribute captures injection activity for applicable wells.|
|iny_gas|❌ Excluded|Constant value (0) across all records. The attribute shows no temporal or well-level variability and provides no analytical value.|
|iny_co2|❌ Excluded|Constant value (0) across all production records. The attribute has no temporal or well-level variability and provides no analytical value.|
|iny_otro|❌ Excluded|Constant value (0) across all production records. The attribute has no temporal or well-level variability and provides no analytical value.|
|tef|✅ fact_production|Dynamic operational measure representing the effective operating time of each well. It varies over time and provides valuable information for production analysis.|
|vida_util|❌ Excluded|The attribute contains 99.83% missing values, and the few available records are all equal to zero, providing no analytical value for the dimensional model.|
|tipoextraccion|✅ fact_production|Operational attribute that describes the extraction method used during each production period. It may change over time for the same well and therefore belongs to the production fact table rather than a static dimension.|
|tipoestado|✅ fact_production|Operational attribute that describes the well status during each production period. It can change over time and therefore belongs to the production fact table rather than a static dimension.|
|tipopozo|✅ fact_production|Operational attribute that describes the well type during each production period. Since it can change over time for the same well, it belongs in the production fact table rather than a static dimension.|

## 9.8 fact_fracturing Attribute Selection

### 9.8.1 Fracture Attribute Validation: `arena_bombeada_nacional_tn`

#### Objective

Evaluate the quality and behavior of the `arena_bombeada_nacional_tn` variable to determine whether it is a valid quantitative measure for the fact_fracturing table.

The analysis will verify:

- Data completeness.
- Presence of missing values.
- Distribution and statistical behavior.
- Presence of zero values.
- Variability between wells.
- Whether the measure changes between different fracturing operations of the same well.

In [340]:
fracture_df["arena_bombeada_nacional_tn"].isna().sum()

np.int64(0)

#### Finding

The arena_bombeada_nacional_tn variable contains no missing values, indicating complete availability across all hydraulic fracturing records.

In [341]:
fracture_df["arena_bombeada_nacional_tn"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, confirming excellent data completeness.

In [342]:
fracture_df["arena_bombeada_nacional_tn"].describe()

count     4806.000000
mean      4629.183169
std       5168.291432
min          0.000000
25%          0.000000
50%       1158.937500
75%       9364.383625
max      20834.000000
Name: arena_bombeada_nacional_tn, dtype: float64

#### Finding

The amount of nationally supplied proppant exhibits high variability across fracturing operations, ranging from 0 to 20,834 tons. The distribution is strongly right-skewed, with a median (1,158.94 tons) substantially lower than the mean (4,629.18 tons), indicating the presence of numerous high-volume treatments.

In [343]:
(fracture_df["arena_bombeada_nacional_tn"] == 0).sum()

np.int64(1565)

In [344]:
(fracture_df["arena_bombeada_nacional_tn"] == 0).mean() * 100

np.float64(32.563462338743236)

#### Finding

A total of 1,565 records (32.56%) contain zero tons of nationally supplied proppant. These values likely represent operations that exclusively used imported sand or treatments where no national proppant was required, rather than missing information.

In [345]:
fracture_df.groupby("idpozo")["arena_bombeada_nacional_tn"].nunique().describe()

count    4562.000000
mean        1.010960
std         0.104127
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
Name: arena_bombeada_nacional_tn, dtype: float64

#### Finding

Most wells present a single value for the amount of nationally supplied proppant. The average of 1.01 distinct values per well indicates that only a small number of wells underwent fracturing operations with different quantities of national sand.

In [346]:
fracture_df.groupby("idpozo")["arena_bombeada_nacional_tn"].nunique().max()

np.int64(2)

#### Finding

The maximum number of distinct values observed for a single well is 2, indicating limited variability across multiple fracturing events.

In [347]:
pozo = (fracture_df.groupby("idpozo")["arena_bombeada_nacional_tn"].nunique())
pozo[pozo > 1]

idpozo
128039    2
135445    2
145002    2
153205    2
155091    2
156622    2
156925    2
157209    2
158105    2
159624    2
159866    2
160206    2
160207    2
160305    2
160306    2
160307    2
160308    2
160465    2
161523    2
161991    2
161998    2
161999    2
162000    2
162001    2
162224    2
162243    2
162356    2
162357    2
162374    2
162456    2
162598    2
162626    2
162629    2
162631    2
162676    2
162702    2
162859    2
162900    2
162932    2
163058    2
163125    2
163149    2
163222    2
163272    2
163370    2
163743    2
163903    2
164062    2
164774    2
164870    2
Name: arena_bombeada_nacional_tn, dtype: int64

#### Finding

Only a small subset of wells presents more than one value for nationally supplied proppant, suggesting that most wells were fractured using a consistent proppant volume, while a limited number experienced modifications between treatments.

In [348]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "arena_bombeada_nacional_tn"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,arena_bombeada_nacional_tn
3050,3684,2022-02-07,0.0
4493,5247,2022-02-07,0.0


#### Finding

The selected well records 0 tons of nationally supplied proppant in both fracturing records, illustrating a case where this material was not used during the treatment.

### Conclusion

The `arena_bombeada_nacional_tn` variable is a complete quantitative measure with no missing values and substantial variability across hydraulic fracturing operations. Although approximately one-third of the records contain zero values, these appear to represent valid operational outcomes rather than missing information. The measure shows only limited variation within the same well, indicating that proppant quantities are generally consistent across repeated treatments.

Therefore, arena_bombeada_nacional_tn should be retained as a core measure in the fact_fracturing table.

### 9.8.2 Fracture Attribute Validation: `arena_bombeada_importada_tn`

#### Objective

Evaluate the quality and behavior of the `arena_bombeada_importada_tn` variable to determine whether it is a valid quantitative measure for the fact_fracturing table.

The analysis will verify:

- Data completeness.
- Presence of missing values.
- Distribution and statistical behavior.
- Presence of zero values.
- Variability between wells.
- Whether the measure changes between different fracturing operations of the same well.

In [349]:
fracture_df["arena_bombeada_importada_tn"].isna().sum()

np.int64(0)

#### Finding

The arena_bombeada_importada_tn variable contains no missing values, indicating complete data availability across all hydraulic fracturing records.

In [350]:
fracture_df["arena_bombeada_importada_tn"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, confirming complete coverage of imported proppant information.

In [351]:
fracture_df["arena_bombeada_importada_tn"].describe()

count     4806.000000
mean       351.662287
std       1779.095270
min          0.000000
25%          0.000000
50%         10.490000
75%        410.321250
max      80652.000000
Name: arena_bombeada_importada_tn, dtype: float64

#### Finding

The amount of imported proppant shows a highly right-skewed distribution, ranging from 0 to 80,652 tons. Although the median is only 10.49 tons, a small number of operations used very large imported sand volumes, producing a mean (351.66 tons) considerably higher than the median.

In [352]:
(fracture_df["arena_bombeada_importada_tn"] == 0).sum()

np.int64(2267)

In [353]:
(fracture_df["arena_bombeada_importada_tn"] == 0).mean() * 100

np.float64(47.170203911776944)

#### Finding

A total of 2,267 records (47.17%) report 0 tons of imported proppant, indicating that imported sand was not used in nearly half of the fracturing operations.

In [354]:
fracture_df.groupby("idpozo")["arena_bombeada_importada_tn"].nunique().describe()

count    4562.000000
mean        1.043402
std         0.263802
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         6.000000
Name: arena_bombeada_importada_tn, dtype: float64

#### Finding

Most wells maintain a single imported proppant value, although the average of 1.04 distinct values per well indicates that some wells experienced changes in imported sand quantities across multiple fracturing operations.

In [355]:
fracture_df.groupby("idpozo")["arena_bombeada_importada_tn"].nunique().max()

np.int64(6)

#### Finding

One well presents up to 6 distinct imported proppant quantities, suggesting greater operational variability than observed for nationally supplied sand.

In [356]:
pozo = (fracture_df.groupby("idpozo")["arena_bombeada_importada_tn"].nunique())
pozo[pozo > 1]

idpozo
6051      2
6187      2
6316      2
7010      2
8322      2
         ..
163272    2
163370    2
164457    3
164458    3
164774    2
Name: arena_bombeada_importada_tn, Length: 149, dtype: int64

#### Finding

Several wells exhibit multiple imported proppant quantities across different fracturing events, indicating that treatment designs may be adjusted over time.

In [357]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "arena_bombeada_importada_tn"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,arena_bombeada_importada_tn
3050,3684,2022-02-07,0.0
4493,5247,2022-02-07,0.0


#### Finding

The selected well records 0 tons of imported proppant in both fracturing events, indicating that imported sand was not used in these treatments.

### Conclusion

The `arena_bombeada_importada_tn` variable is a complete quantitative measure with no missing values and substantial operational variability. Nearly half of the fracturing operations did not use imported proppant, while others employed very large imported sand volumes, resulting in a highly skewed distribution. Compared with nationally supplied sand, imported proppant exhibits greater variability between repeated treatments of the same well.

Therefore, arena_bombeada_importada_tn should be retained as a core measure in the fact_fracturing table.

### 9.8.3 Fracture Attribute Validation: `agua_inyectada_m3`

#### Objective

Evaluate the quality and behavior of the `agua_inyectada_m3` variable to determine whether it should be retained as a quantitative measure in the fact_fracturing table.

The analysis will verify:

- Data completeness.
- Presence of missing values.
- Distribution of injected water volumes.
- Presence of zero values.
- Variability between wells.
- Whether injected water changes across multiple fracturing operations of the same well.

In [358]:
fracture_df["agua_inyectada_m3"].isna().sum()

np.int64(0)

#### Finding

The agua_inyectada_m3 variable contains no missing values, indicating complete data availability for every hydraulic fracturing record.

In [359]:
fracture_df["agua_inyectada_m3"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, confirming full coverage of injected water volume across the dataset.

In [360]:
fracture_df["agua_inyectada_m3"].describe()

count      4806.000000
mean      31792.340439
std       36281.220097
min           0.000000
25%         420.000000
50%       17151.350000
75%       59537.102494
max      537184.600000
Name: agua_inyectada_m3, dtype: float64

#### Finding

Injected water volumes exhibit a highly right-skewed distribution, ranging from 0 to 537,184.6 m³. The median (17,151.35 m³) is considerably lower than the mean (31,792.34 m³), indicating the presence of very large hydraulic fracturing treatments that increase the average.

In [361]:
(fracture_df["agua_inyectada_m3"] == 0).sum()

np.int64(305)

In [362]:
(fracture_df["agua_inyectada_m3"] == 0).mean() * 100

np.float64(6.346233874323762)

#### Finding

A total of 305 records (6.35%) report 0 m³ of injected water. This represents a relatively small proportion of the dataset, suggesting that nearly all recorded fracturing operations involved water injection.

In [363]:
fracture_df.groupby("idpozo")["agua_inyectada_m3"].nunique().describe()

count    4562.000000
mean        1.040552
std         0.256252
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         6.000000
Name: agua_inyectada_m3, dtype: float64

#### Finding

Most wells present a single injected water volume, while the average of 1.04 distinct values per well indicates that some wells underwent multiple fracturing operations with different water requirements.

In [364]:
fracture_df.groupby("idpozo")["agua_inyectada_m3"].nunique().max()

np.int64(6)

#### Finding

One well records up to 6 distinct injected water volumes, reflecting considerable variability in fracturing treatment design across repeated operations.

In [365]:
pozo = (fracture_df.groupby("idpozo")["agua_inyectada_m3"].nunique())
pozo[pozo > 1]

idpozo
6051      2
6316      2
68525     2
126501    2
128039    4
         ..
164062    2
164457    2
164458    3
164774    2
164870    2
Name: agua_inyectada_m3, Length: 139, dtype: int64

#### Finding

Several wells exhibit different injected water volumes across multiple fracturing events, demonstrating that treatment parameters were adjusted over time according to operational requirements.

In [366]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "agua_inyectada_m3"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,agua_inyectada_m3
3050,3684,2022-02-07,0.0
4493,5247,2022-02-07,0.0


#### Finding

The selected well reports 0 m³ of injected water in both fracturing records, indicating no recorded water injection for these treatments.

### Conclusion

The `agua_inyectada_m3` variable is a complete quantitative measure with no missing values and substantial operational variability. Water injection is present in the vast majority of hydraulic fracturing operations, while only a small percentage of records report zero values. The observed variability between repeated treatments of the same well reflects changes in fracturing design rather than data inconsistencies.

Therefore, agua_inyectada_m3 should be retained as a core measure in the fact_fracturing table.

### 9.8.4 Fracture Attribute Validation: `co2_inyectado_m3`

#### Objective

Evaluate the quality and behavior of the `co2_inyectado_m3` variable to determine whether it should be retained as a quantitative measure in the fact_fracturing table.

In [367]:
fracture_df["co2_inyectado_m3"].isna().sum()

np.int64(0)

#### Finding

The co2_inyectado_m3 variable contains no missing values, indicating complete data availability across all hydraulic fracturing records.

In [368]:
fracture_df["co2_inyectado_m3"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, confirming that CO₂ injection was systematically recorded, regardless of whether it was used.

In [369]:
fracture_df["co2_inyectado_m3"].describe()

count    4806.000000
mean        5.562957
std        37.750349
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       560.000000
Name: co2_inyectado_m3, dtype: float64

#### Finding

Injected CO₂ volumes range from 0 to 560 m³, with a mean of 5.56 m³ and a median of 0 m³. This distribution indicates that CO₂ injection was only employed in a limited number of hydraulic fracturing operations.

In [370]:
(fracture_df["co2_inyectado_m3"] == 0).sum()

np.int64(4647)

In [371]:
(fracture_df["co2_inyectado_m3"] == 0).mean() * 100

np.float64(96.6916354556804)

#### Finding

A total of 4,647 records (96.69%) report 0 m³ of injected CO₂, showing that CO₂ was absent from the vast majority of treatments.

In [372]:
fracture_df.groupby("idpozo")["co2_inyectado_m3"].nunique().describe()

count    4562.000000
mean        1.002630
std         0.088804
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         6.000000
Name: co2_inyectado_m3, dtype: float64

#### Finding

Nearly every well records a single CO₂ injection value. The average of 1.00 distinct values per well indicates extremely limited variability across repeated fracturing operations.

In [373]:
fracture_df.groupby("idpozo")["co2_inyectado_m3"].nunique().max()

np.int64(6)

#### Finding

The maximum number of distinct CO₂ injection volumes observed for a single well is 6, indicating that a small number of wells experienced different CO₂ injection strategies over multiple treatments.

In [374]:
pozo = (fracture_df.groupby("idpozo")["co2_inyectado_m3"].nunique())
pozo[pozo > 1]

idpozo
136630    3
159341    6
159889    2
160084    2
160102    2
161465    3
Name: co2_inyectado_m3, dtype: int64

#### Finding

Only a handful of wells present multiple CO₂ injection values, confirming that CO₂ utilization is restricted to a limited subset of hydraulic fracturing operations.

In [375]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "co2_inyectado_m3"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,co2_inyectado_m3
3050,3684,2022-02-07,0.0
4493,5247,2022-02-07,0.0


#### Finding

The selected well records 0 m³ of injected CO₂ in both fracturing events, illustrating the typical behavior observed throughout the dataset.

### Conclusion

The `co2_inyectado_m3` variable is a complete quantitative measure with no missing values. Although 96.69% of the records contain zero values, these represent valid operational outcomes rather than missing information, indicating that CO₂ injection is an optional component used only in specific hydraulic fracturing treatments. The limited variability observed between repeated operations further supports this interpretation.

Therefore, co2_inyectado_m3 should be retained as a quantitative measure in the fact_fracturing table, since it captures an important treatment characteristic whenever CO₂ is employed.

### 9.8.5 Fracture Attribute Validation: `presion_maxima_psi`

#### Objective

Evaluate the quality and behavior of the `presion_maxima_psi` variable to determine whether it should be retained as a quantitative measure in the fact_fracturing table.


In [376]:
fracture_df["presion_maxima_psi"].isna().sum()

np.int64(0)

#### Finding

The presion_maxima_psi variable contains no missing values, indicating complete recording of maximum treatment pressure for all hydraulic fracturing operations.

In [377]:
fracture_df["presion_maxima_psi"].isna().mean() * 100

np.float64(0.0)

#### Finding

The percentage of missing values is 0%, confirming complete data availability.

In [378]:
fracture_df["presion_maxima_psi"].describe()

count      4806.000000
mean       9238.380996
std        6335.734303
min           0.000000
25%        6763.250000
50%       10900.529569
75%       11769.105414
max      209640.559334
Name: presion_maxima_psi, dtype: float64

#### Finding

Maximum treatment pressure ranges from 0 psi to 209,640.56 psi, with a mean of 9,238.38 psi and a median of 10,900.53 psi. The wide dispersion suggests substantial variability among hydraulic fracturing operations. The extremely high maximum value should be reviewed during the data quality assessment, as it may represent an exceptional operation or a potential outlier.

In [379]:
(fracture_df["presion_maxima_psi"] == 0).sum()

np.int64(76)

In [380]:
(fracture_df["presion_maxima_psi"] == 0).mean() * 100

np.float64(1.581356637536413)

#### Finding

Only 76 records (1.58%) report a pressure of 0 psi, indicating that almost every fracturing operation includes a valid maximum pressure measurement.

In [381]:
fracture_df.groupby("idpozo")["presion_maxima_psi"].nunique().describe()

count    4562.000000
mean        1.046252
std         0.265391
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         6.000000
Name: presion_maxima_psi, dtype: float64

#### Finding

Most wells record a single maximum pressure value, with an average of 1.05 distinct values per well, indicating limited variability between repeated fracturing operations.

In [382]:
fracture_df.groupby("idpozo")["presion_maxima_psi"].nunique().max()

np.int64(6)

#### Finding

The highest variability observed for an individual well is 6 distinct pressure values, showing that some wells underwent multiple fracturing treatments with different operational conditions.

In [383]:
pozo = (fracture_df.groupby("idpozo")["presion_maxima_psi"].nunique())
pozo[pozo > 1]

idpozo
6051      2
6187      2
6316      2
7010      2
8322      2
         ..
164062    2
164457    3
164458    3
164774    2
164870    2
Name: presion_maxima_psi, Length: 165, dtype: int64

#### Finding

A subset of wells presents multiple maximum pressure values, confirming that treatment pressure can legitimately vary across successive hydraulic fracturing operations.

In [384]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "presion_maxima_psi"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,presion_maxima_psi
3050,3684,2022-02-07,7700.0
4493,5247,2022-02-07,7700.0


#### Finding

The selected well records the same maximum pressure (7,700 psi) in both fracturing events, illustrating that repeated operations may also maintain identical treatment conditions.

### Conclusion

The `presion_maxima_psi` variable is a high-quality quantitative measure, with complete coverage and only 1.58% of records equal to zero. It captures an essential operational characteristic of hydraulic fracturing and shows realistic variability across treatments. Although an exceptionally high value (209,640.56 psi) was identified, it should be flagged for future outlier analysis rather than removed during the data audit.

### 9.8.6 Fracture Attribute Validation: `potencia_equipos_fractura_hp`

#### Objective

Evaluate the quality and behavior of the `potencia_equipos_fractura_hp` variable to determine whether it should be retained as a quantitative measure in the fact_fracturing table.

In [385]:
fracture_df["potencia_equipos_fractura_hp"].isna().sum()

np.int64(51)

#### Finding

The potencia_equipos_fractura_hp variable contains 51 missing values, representing a very small fraction of the dataset.

In [386]:
fracture_df["potencia_equipos_fractura_hp"].isna().mean() * 100

np.float64(1.0611735330836454)

#### Finding

Missing values account for 1.06% of all hydraulic fracturing records, indicating high overall completeness.

In [387]:
fracture_df["potencia_equipos_fractura_hp"].describe()

count      4755.000000
mean      20680.067371
std       15667.555303
min           0.000000
25%        2198.500000
50%       22062.000000
75%       34000.000000
max      232159.289200
Name: potencia_equipos_fractura_hp, dtype: float64

#### Finding

Equipment power ranges from 0 HP to 232,159.29 HP, with a mean of 20,680.07 HP and a median of 22,062 HP. The broad distribution reflects the diversity of fracturing operations. The extremely high maximum value should be reviewed during the outlier detection stage, as it may represent an exceptional treatment or a recording anomaly.

In [388]:
(fracture_df["potencia_equipos_fractura_hp"] == 0).sum()

np.int64(154)

In [389]:
(fracture_df["potencia_equipos_fractura_hp"] == 0).mean() * 100

np.float64(3.204327923429047)

#### Finding

A total of 154 records (3.20%) contain 0 HP, indicating that almost all fracturing treatments report a valid equipment power value.

In [390]:
fracture_df.groupby("idpozo")["potencia_equipos_fractura_hp"].nunique().describe()

count    4562.000000
mean        1.034853
std         0.251048
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         6.000000
Name: potencia_equipos_fractura_hp, dtype: float64

#### Finding

Most wells present a single equipment power value across their recorded treatments. The average of 1.03 distinct values per well indicates limited variability between repeated fracturing operations.

In [391]:
fracture_df.groupby("idpozo")["potencia_equipos_fractura_hp"].nunique().max()

np.int64(6)

#### Finding

The maximum number of distinct equipment power values observed for a single well is 6, indicating that some wells underwent multiple fracturing operations with different equipment configurations.

In [392]:
pozo = (fracture_df.groupby("idpozo")["potencia_equipos_fractura_hp"].nunique())
pozo[pozo > 1]

idpozo
6051      2
6187      2
6316      2
7010      2
8322      2
         ..
164062    2
164457    3
164458    3
164774    2
164870    2
Name: potencia_equipos_fractura_hp, Length: 148, dtype: int64

#### Finding

Several wells exhibit multiple equipment power values, confirming that different fracturing stages or interventions may use different pumping capacities.

In [393]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "potencia_equipos_fractura_hp"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,potencia_equipos_fractura_hp
3050,3684,2022-02-07,2574.0
4493,5247,2022-02-07,0.0


#### Finding

The selected well presents two fracturing records on the same date with different equipment power values (2,574 HP and 0 HP). Since the remaining operational variables for these records also differ in some cases, the zero value likely represents either a missing operational measurement or a specific recording condition rather than a duplicated event. This example reinforces the need to preserve the variable while considering zero values during future data quality assessments.

### Conclusion

The `potencia_equipos_fractura_hp` variable is a high-quality quantitative measure with 98.94% completeness. Although a small number of records contain missing or zero values, the variable captures an essential operational characteristic of hydraulic fracturing by describing the pumping power available during each treatment. The observed variability across repeated operations demonstrates that equipment configurations may legitimately change over time. An exceptionally large maximum value (232,159.29 HP) should be reviewed during the outlier detection stage but should not be modified during the data audit.

### 9.8.7 Fracture Attribute Validation: `cantidad_fracturas`

#### Objective

To evaluate the quality and behavior of the variable `cantidad_fracturas` to determine if it can be incorporated as a measure in the fact_fracturing table. The presence of missing values, data distribution, the existence of null (0) values, variability between treatments within the same well, and the temporal consistency of the logs will be verified.

In [394]:
fracture_df["cantidad_fracturas"].isna().sum()

np.int64(0)

In [395]:
fracture_df["cantidad_fracturas"].isna().mean() * 100

np.float64(0.0)

#### Finding

The cantidad_fracturas variable contains no missing values, achieving 100% completeness across all hydraulic fracturing records.

In [396]:
fracture_df["cantidad_fracturas"].describe()

count    4806.000000
mean       22.608198
std        21.174029
min         1.000000
25%         3.000000
50%        16.000000
75%        41.000000
max       105.000000
Name: cantidad_fracturas, dtype: float64

#### Finding

The number of fracturing stages ranges from 1 to 105, with a mean of 22.61 stages and a median of 16 stages. The wide distribution reflects substantial operational variability between fracturing treatments.

In [397]:
(fracture_df["cantidad_fracturas"] == 0).sum()

np.int64(0)

In [398]:
(fracture_df["cantidad_fracturas"] == 0).mean() * 100

np.float64(0.0)

#### Finding

No records contain zero fracturing stages, indicating that every registered treatment includes at least one completed fracturing stage.

In [399]:
fracture_df.groupby("idpozo")["cantidad_fracturas"].nunique().describe()

count    4562.000000
mean        1.027839
std         0.178586
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         4.000000
Name: cantidad_fracturas, dtype: float64

#### Finding

Most wells exhibit a single value for the number of fracturing stages. The average of 1.03 distinct values per well indicates that this operational characteristic remains highly consistent across repeated treatments of the same well.

In [400]:
fracture_df.groupby("idpozo")["cantidad_fracturas"].nunique().max()

np.int64(4)

#### Finding

The highest variability observed for a single well is 4 distinct values, demonstrating that some wells experienced multiple fracturing operations with different numbers of stages.

In [401]:
pozo = (fracture_df.groupby("idpozo")["cantidad_fracturas"].nunique())
pozo[pozo > 1]

idpozo
6051      2
6187      2
6316      2
8322      2
92258     2
         ..
163058    2
163149    2
163370    2
163743    2
163903    2
Name: cantidad_fracturas, Length: 117, dtype: int64

#### Finding

A limited number of wells present multiple values for the number of fracturing stages, suggesting that repeated interventions may involve different completion designs or operational strategies.

In [402]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "cantidad_fracturas"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,cantidad_fracturas
3050,3684,2022-02-07,2
4493,5247,2022-02-07,2


#### Finding

The selected well records two fracturing events on the same date, both with 2 fracturing stages, confirming consistency for this operational measure within repeated records.

### Conclusion

The `cantidad_fracturas` variable demonstrates excellent data quality, with 100% completeness, no zero values, and consistent behavior across wells. The observed variability reflects legitimate differences in fracturing designs rather than data inconsistencies. Since the number of fracturing stages is one of the primary operational characteristics of a hydraulic fracturing treatment, this variable should be preserved as a core quantitative measure within the fact_fracturing table.

### 9.8.8 Fracture Attribute Validation: `longitud_rama_horizontal_m`

#### Objective

To evaluate the quality and behavior of the variable `longitud_rama_horizontal_m` to determine whether it represents an operational measure associated with each fracturing treatment or a relatively stable physical attribute of the well. Its completeness, distribution, presence of null values, consistency between treatments within the same well, and temporal variability will be analyzed.

In [403]:
fracture_df["longitud_rama_horizontal_m"].isna().sum()

np.int64(0)

In [404]:
fracture_df["longitud_rama_horizontal_m"].isna().mean() * 100

np.float64(0.0)

#### Finding

The longitud_rama_horizontal_m variable contains no missing values, achieving 100% completeness across all hydraulic fracturing records.

In [405]:
fracture_df["longitud_rama_horizontal_m"].describe()

count    4806.000000
mean     1309.689564
std      1351.789993
min         0.000000
25%         0.000000
50%      1135.750000
75%      2540.750000
max      5114.110000
Name: longitud_rama_horizontal_m, dtype: float64

#### Finding

The horizontal lateral length ranges from 0 m to 5,114.11 m, with a mean of 1,309.69 m and a median of 1,135.75 m. The distribution indicates considerable variability among wells, reflecting different drilling designs and reservoir development strategies.

In [406]:
(fracture_df["longitud_rama_horizontal_m"] == 0).sum()

np.int64(2050)

In [407]:
(fracture_df["longitud_rama_horizontal_m"] == 0).mean() * 100

np.float64(42.655014565126926)

#### Finding

Approximately 42.66% of the records have a horizontal length equal to 0 m. This high proportion suggests that zero likely represents wells without a horizontal branch (e.g., conventional vertical wells) or cases where the horizontal length is not applicable, rather than a data quality issue.

In [408]:
fracture_df.groupby("idpozo")["longitud_rama_horizontal_m"].nunique().describe()

count    4562.000000
mean        1.011179
std         0.145404
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         5.000000
Name: longitud_rama_horizontal_m, dtype: float64

#### Finding

Most wells have a single recorded horizontal length. The average of 1.01 distinct values per well indicates that this characteristic remains essentially constant across repeated fracturing treatments.

In [409]:
fracture_df.groupby("idpozo")["longitud_rama_horizontal_m"].nunique().max()

np.int64(5)

#### Finding

The maximum variability observed for a single well is 5 distinct values, indicating that only a small number of wells present changes in the recorded horizontal length across different records.

In [410]:
pozo = (fracture_df.groupby("idpozo")["longitud_rama_horizontal_m"].nunique())
pozo[pozo > 1]

idpozo
68525     2
130158    2
130518    2
131344    3
131431    2
135910    3
136448    2
136581    3
136856    5
137715    2
145002    2
145171    3
145328    4
145468    2
145585    2
145968    2
147024    3
147505    4
152779    4
152786    3
160008    3
160308    2
160770    2
161991    2
161998    2
161999    2
162000    2
162001    2
162243    2
162505    2
162676    2
162900    2
164457    2
164458    3
Name: longitud_rama_horizontal_m, dtype: int64

#### Finding

Only a limited subset of wells exhibits more than one recorded horizontal length. These cases should be reviewed individually to determine whether they correspond to data corrections, updated measurements, or operational changes.

In [411]:
fracture_df.loc[
    fracture_df["idpozo"] == 160161,
    ["id_base_fractura_adjiv", "fecha_inicio_fractura", "longitud_rama_horizontal_m"]
].sort_values("fecha_inicio_fractura")

,id_base_fractura_adjiv,fecha_inicio_fractura,longitud_rama_horizontal_m
3050,3684,2022-02-07,0.0
4493,5247,2022-02-07,0.0


#### Finding

The selected well records two fracturing events on the same date, both with a horizontal length of 0 m, confirming consistency for this attribute within repeated records.

### Conclusion

The `longitud_rama_horizontal_m` variable presents excellent data completeness and remains highly consistent across treatments for the same well, suggesting that it describes a physical characteristic of the well rather than an operational characteristic of a specific fracturing treatment. Although it can be retained in fact_fracturing for analytical convenience, its behavior indicates that it is conceptually better suited as an attribute of the well.

|Column|Decision|Reason|
|------|--------|------|
|arena_bombeada_nacional_tn|✅ fact_fracturing|Quantitative measure describing the amount of nationally supplied proppant used during each hydraulic fracturing operation. It is complete, operationally meaningful, and directly characterizes the treatment performed.|
|arena_bombeada_importada_tn|✅ fact_fracturing|Quantitative measure describing the amount of imported proppant used during each hydraulic fracturing operation. It is complete, operationally relevant, and captures treatment design characteristics.|
|agua_inyectada_m3|✅ fact_fracturing|Quantitative measure representing the volume of water injected during each hydraulic fracturing treatment. It is complete, operationally meaningful, and captures treatment intensity.|
|co2_inyectado_m3|✅ fact_fracturing|Quantitative measure representing the volume of CO₂ injected during hydraulic fracturing. Despite the high proportion of zero values, it captures a valid operational characteristic for treatments where CO₂ is used.|
|presion_maxima_psi|✅ fact_fracturing|Quantitative measure representing the maximum pressure reached during hydraulic fracturing. It has complete coverage, very few zero values, and captures an essential operational characteristic of each treatment.|
|potencia_equipos_fractura_hp|✅ fact_fracturing|Quantitative measure representing the pumping power available during hydraulic fracturing. It provides essential operational information and exhibits high completeness despite a small proportion of missing values.|
|cantidad_fracturas|✅ fact_fracturing|Core quantitative measure representing the number of fracturing stages performed during each treatment. The variable exhibits excellent completeness, no invalid zero values, and strong operational consistency.|
|longitud_rama_horizontal_m|⚠️ Preferably dim_well (or equivalent well dimension)|Represents a stable physical characteristic of the well rather than an operational measurement of a fracturing treatment. It remains nearly constant across interventions and therefore fits better as a descriptive attribute than as a fact measure.|

## 10. Final Dimensional Model

### Notebook Overview

This section presents the final dimensional model definition resulting from the complete source data audit and column-level evaluation performed throughout this notebook.

The objective is to transform the original operational datasets into an analytical model following a dimensional data warehouse approach, separating:

- descriptive attributes that define business entities (dimensions),
- measurable operational events (facts),
- and temporal references required for analytical analysis.

The final model is designed to support production performance analysis, hydraulic fracturing evaluation, geological segmentation, company comparison, and operational decision-making.

### 10.1 Dimensional Model Overview

Based on the previous attribute evaluation, the final data warehouse model will consist of the following components:


#### 10.1.1 Dimensions

| Dimension | Purpose |
|-----------|---------|
| dim_time | Provides a standardized temporal reference for production and fracturing events. |
| dim_well | Stores stable physical and operational-independent characteristics of each well. |
| dim_geology | Contains geological attributes used for reservoir and formation segmentation. |
| dim_location | Stores geographical information related to well location. |
| dim_field_area | Contains concession and producing field information. |

### 10.1.2 Fact Tables

| Fact Table | Purpose |
|------------|---------|
| fact_production | Stores monthly production measurements and operational conditions of each well. |
| fact_fracturing | Stores hydraulic fracturing treatment measurements and operational parameters. |

## 10.2 Dimension Tables Definition

### dim_time

#### Purpose

The Time Dimension provides a unified temporal structure for all analytical processes.

Production records are monthly-based and use `fecha_data` as the reporting date, while fracturing records contain operational dates related to treatment execution.

#### Main Attributes

| Column | Description |
|--------|-------------|
| date_key | Surrogate key generated during ETL. |
| full_date | Complete calendar date used for relationships. |
| year | Calendar year. |
| month | Calendar month. |
| quarter | Calendar quarter. |
| month_name | Month descriptive name. |

#### Business Decision

Business dates will be used as relationships with fact tables, while administrative dates such as data loading timestamps will remain outside the analytical model and will only support data lineage if required.

### dim_well

#### Purpose

Stores stable characteristics associated with each well.

#### Main Attributes

| Column |
|--------|
| idpozo |
| sigla |
| profundidad |
| proyecto |
| clasificacion |
| subclasificacion |
| longitud_rama_horizontal_m |

#### Business Decision

Only attributes representing stable well characteristics are included.

Operational attributes such as extraction method, well status, and well type were excluded because they change over time and belong to fact_production.

### dim_geology

#### Purpose

Stores geological characteristics used for reservoir and formation analysis.

#### Main Attributes

| Column |
|--------|
| formacion |
| subtipo_recurso |
| tipo_reservorio |
| subtipo_reservorio |
| formacion_productiva |

#### Business Decision

Geological attributes are considered descriptive and relatively stable. They provide meaningful analytical segmentation between formations, reservoirs, and unconventional resource types.

### dim_location

#### Purpose

Stores geographical information related to wells.

#### Main Attributes

| Column |
|--------|
| cuenca |
| provincia |
| coordenadax |
| coordenaday |

#### Business Decision

Location attributes represent stable geographical characteristics and allow regional production analysis.

### dim_field_area

#### Purpose

Stores concession and producing field information.

#### Main Attributes

| Column |
|--------|
| idareapermisoconcesion |
| areapermisoconcesion |
| idareayacimiento |
| areayacimiento |

#### Business Decision

These attributes represent stable field-level information associated with wells.

### dim_company

#### Purpose

Stores operator information.

#### Main Attributes

| Column |
|--------|
| idempresa |
| empresa |

#### Business Decision

Company information represents a stable business entity and will be maintained as a separate dimension.

## 10.3 Fact Tables Definition

### fact_production

#### Purpose

Stores monthly production measurements and operational conditions for each well.

#### Grain Definition

One record represents:

**One well (`idpozo`) measured during one production month.**

### Main Attributes

#### Measures

| Column |
|--------|
| prod_pet |
| prod_gas |
| prod_agua |
| iny_agua |
| tef |

#### Operational Attributes

| Column |
|--------|
| tipoextraccion |
| tipoestado |
| tipopozo |

#### Business Decision

Operational attributes are included in the fact table because they represent conditions that can change over time.

The table will be linked to:

- dim_well through `idpozo`
- dim_time through the production reporting date.

### fact_fracturing

#### Purpose

Stores hydraulic fracturing treatment measurements and operational parameters.

### Grain Definition

One record represents:

**One hydraulic fracturing operation performed on one well.**

#### Main Attributes

| Column |
|--------|
| arena_bombeada_nacional_tn |
| arena_bombeada_importada_tn |
| agua_inyectada_m3 |
| co2_inyectado_m3 |
| presion_maxima_psi |
| potencia_equipos_fractura_hp |
| cantidad_fracturas |

#### Business Decision

These attributes represent quantitative characteristics of each fracturing treatment and are therefore modeled as fact measures.

`longitud_rama_horizontal_m` was not included as a fact measure because it represents a physical well characteristic rather than a treatment measurement. It will be maintained as part of the well dimension.

The table will be linked to:

- dim_well through `idpozo`
- dim_time through fracturing business dates.

## 10.4 Excluded Attributes

Several attributes were excluded from the dimensional model because they did not provide analytical value.

| Attribute | Reason |
|-----------|--------|
| tipo_de_recurso | Constant value across the dataset. |
| rectificado | Constant value across the dataset. |
| habilitado | Constant value across the dataset. |
| observaciones | Mostly null and contains automatic operational messages. |
| fechaingreso | Administrative attribute that changes over time and does not describe the well. |
| formprod | Redundant coded representation of geological information. |
| iny_gas | Constant value equal to zero. |
| iny_co2 | Constant value equal to zero in production records. |
| iny_otro | Constant value equal to zero in production records. |
| vida_util | Extremely high missingness and no analytical variation. |

## 10.5 Final Model Structure

The final dimensional model follows a star schema approach where fact tables represent operational events and dimensions provide descriptive context for analytical analysis.

Both fact tables share common dimensions, allowing integrated analysis across production and hydraulic fracturing activities.

```mermaid
flowchart TB

    DT[dim_time]
    DW[dim_well]
    DG[dim_geology]
    DL[dim_location]
    DFA[dim_field_area]
    DC[dim_company]

    FP[fact_production]
    FF[fact_fracturing]

    DT --> FP
    DT --> FF

    DW --> FP
    DW --> FF

    DG --> FP
    DG --> FF

    DL --> FP
    DL --> FF

    DFA --> FP
    DFA --> FF

    DC --> FP
    DC --> FF
```

#### Model Relationships

|Fact Table|Related Dimensions|
|----------|------------------|
|fact_production|dim_time, dim_well, dim_geology, dim_location, dim_field_area, dim_company|
|fact_fracturing|dim_time, dim_well, dim_geology, dim_location, dim_field_area, dim_company|

## 10.6 Final Conclusion

The dimensional modeling process successfully transformed the original operational datasets into a structured analytical model.

The final design separates:

- stable business entities into dimensions,
- measurable operational events into fact tables,
- and temporal references into a centralized Time Dimension.

This structure enables scalable analytical workflows, avoids redundancy, preserves historical operational changes, and supports future SQL-based analysis and BI reporting.

The model is now ready for the next stage: database implementation and ETL development in MySQL.

## 10.5 Final Model Structure

The final dimensional model is summarized as follows: